# SQL Server Query Writing II
### Sorting, limiting, random sampling, unique identifiers, SELECT INTO and set operators

This notebook is a structured, runnable companion to the instructor's notes for the *Query Writing 2* session. Every topic follows the same four-part pattern:

1. **Purpose** – what the feature is for and why you would reach for it.
2. **Syntax** – the general shape of the command.
3. **Examples** – small, runnable cells, each with a note on what to look for in the output.
4. **Pitfalls** – the mistakes and misunderstandings that most often cost people time.

**Order of topics (it follows the order of the instructor's notes):**

| # | Topic | Instructor's keywords |
|---|-------|-----------------------|
| 0 | Setup: running T-SQL from Jupyter | `USE`, `GO` |
| 1 | `ORDER BY` | ASC / DESC, `ISNULL`, alias, ordinal position |
| 2 | `TOP N` | `TOP`, `WITH TIES`, `PERCENT`, ranking |
| 3 | Random sampling and unique identifiers | `TOP 70 PERCENT`, `IDENTITY`, GUID, `NEWID()`, `NEWSEQUENTIALID()` |
| 4 | Fully qualified names | `pubs..titles` |
| 5 | Making-table queries | `SELECT ... INTO` |
| 6 | Set operators | `UNION ALL`, `UNION`, `INTERSECT`, `EXCEPT` |
| 7 | Cheat sheet, fixes for the notes, practice | |

**Databases used:** `pubs` (a small publishing sample: books, authors, stores, publishers) and `AdventureWorksDW2022` (a data-warehouse sample with 18,484 customers in `dbo.DimCustomer`).

## 0. How to read this notebook

**What you are looking at.** Every code cell holds T-SQL exactly as you would type it in SSMS or Azure Data Studio, and the output underneath is the stored result. You do not need to run anything, and no SQL kernel is attached on purpose. If JupyterLab asks you to pick a kernel, choose *No Kernel*.

**Where the outputs come from.** I could not run the queries on a live SQL Server. Instead, each output was produced by replaying the query's logic on Microsoft's published sample data: `pubs` from `instpubs.sql` (23 authors, 8 publishers, 18 titles, 6 stores) and `DimCustomer.csv` from AdventureWorksDW (18,484 customers). Please read the outputs with these conventions in mind:

- **Order without `ORDER BY`.** Where a query has no `ORDER BY`, SQL Server does not guarantee any order. The output shows the order you would most likely see (clustered-index order), and the caption says so.
- **Random values.** Cells that use `NEWID()` or `TABLESAMPLE` are marked *random*. Their output is one example run; yours will differ every time.
- **Dates.** Ages are computed as of 3 October 2026, because `GETDATE()` changes the answer. Two books (`MC3026`, `PC9999`) have no publication date in the install script, so SQL Server stores the day you installed `pubs`; here that is shown as 2026-10-03.
- **Number formats.** Prices are `MONEY`, so they appear with four decimals (19.9900), as in SSMS.
- **Error cells.** The three cells that fail on purpose show SQL Server's standard error text.
- **`USE` and `GO`.** `USE database;` selects the database. The instructor's `GO` (batch separator) is understood only by SSMS, so cells here do not use it.
- If your copy of `pubs` is an older or modified version, data types (for example `tid` instead of `CHAR(6)`) or a few rows may differ.

---
## 1. `ORDER BY` — controlling the order of the result

### Purpose
A table is a *set* of rows, and a set has no order. Unless you say otherwise, SQL Server returns rows in whatever order is convenient for it, and that order can change from one run to the next. `ORDER BY` is the **only** way to guarantee the order of a result. The instructor's note calls this the *display order* (ترتیب نمایش): it changes how the result is presented, never what is stored.

### Syntax
```sql
SELECT   column_list
FROM     schema.table AS alias
ORDER BY expression_1 [ASC | DESC],
         expression_2 [ASC | DESC], ...
```

### Where `ORDER BY` sits in the processing order
The instructor's skeleton `SELECT DISTINCT ... FROM ... ORDER BY` is the order in which you *write* the clauses. SQL Server *processes* them in a different order:

`FROM` → `WHERE` → `GROUP BY` → `HAVING` → `SELECT` → `DISTINCT` → `ORDER BY` → `TOP`

Because `ORDER BY` runs after `SELECT`, it can use column aliases created in the `SELECT` list. `WHERE` runs before `SELECT`, so it cannot. This one fact explains several examples below.

### Key rules
- Default direction is **ASC** (ascending, صعودی). Write `DESC` for descending.
- Sort by several columns by separating them with commas; later columns only break ties left by earlier ones.
- In SQL Server, `NULL` counts as the *lowest* value: it comes first in ASC and last in DESC.
- You may sort by an expression, a column that is not in the `SELECT` list, an alias, or a column position number.

**Example 1.1 — look at the table first.** `titles` has 18 books; the columns we use are `title_id`, `title`, `type`, `price` and `pubdate`. Notice that a couple of books have no price (`NULL`).

In [ ]:
USE pubs;

SELECT *
FROM dbo.titles AS t;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title_id</th>
      <th>title</th>
      <th>type</th>
      <th>pub_id</th>
      <th>price</th>
      <th>advance</th>
      <th>royalty</th>
      <th>ytd_sales</th>
      <th>notes</th>
      <th>pubdate</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>BU1032</td>
      <td>The Busy Executive's Database Guide</td>
      <td>business</td>
      <td>1389</td>
      <td>19.9900</td>
      <td>5000.0000</td>
      <td>10</td>
      <td>4095</td>
      <td>An overview of available database systems with emphasis on common business applications. Illustrated.</td>
      <td>1991-06-12 00:00:00.000</td>
    </tr>
    <tr>
      <td>BU1111</td>
      <td>Cooking with Computers: Surreptitious Balance Sheets</td>
      <td>business</td>
      <td>1389</td>
      <td>11.9500</td>
      <td>5000.0000</td>
      <td>10</td>
      <td>3876</td>
      <td>Helpful hints on how to use your electronic resources to the best advantage.</td>
      <td>1991-06-09 00:00:00.000</td>
    </tr>
    <tr>
      <td>BU2075</td>
      <td>You Can Combat Computer Stress!</td>
      <td>business</td>
      <td>0736</td>
      <td>2.9900</td>
      <td>10125.0000</td>
      <td>24</td>
      <td>18722</td>
      <td>The latest medical and psychological techniques for living with the electronic office. Easy-to-understand explanations.</td>
      <td>1991-06-30 00:00:00.000</td>
    </tr>
    <tr>
      <td>BU7832</td>
      <td>Straight Talk About Computers</td>
      <td>business</td>
      <td>1389</td>
      <td>19.9900</td>
      <td>5000.0000</td>
      <td>10</td>
      <td>4095</td>
      <td>Annotated analysis of what computers can do for you: a no-hype guide for the critical user.</td>
      <td>1991-06-22 00:00:00.000</td>
    </tr>
    <tr>
      <td>MC2222</td>
      <td>Silicon Valley Gastronomic Treats</td>
      <td>mod_cook</td>
      <td>0877</td>
      <td>19.9900</td>
      <td>0.0000</td>
      <td>12</td>
      <td>2032</td>
      <td>Favorite recipes for quick, easy, and elegant meals.</td>
      <td>1991-06-09 00:00:00.000</td>
    </tr>
    <tr>
      <td>MC3021</td>
      <td>The Gourmet Microwave</td>
      <td>mod_cook</td>
      <td>0877</td>
      <td>2.9900</td>
      <td>15000.0000</td>
      <td>24</td>
      <td>22246</td>
      <td>Traditional French gourmet recipes adapted for modern microwave cooking.</td>
      <td>1991-06-18 00:00:00.000</td>
    </tr>
    <tr>
      <td>MC3026</td>
      <td>The Psychology of Computer Cooking</td>
      <td>UNDECIDED</td>
      <td>0877</td>
      <td>NULL</td>
      <td>NULL</td>
      <td>NULL</td>
      <td>NULL</td>
      <td>NULL</td>
      <td>2026-10-03 00:00:00.000</td>
    </tr>
    <tr>
      <td>PC1035</td>
      <td>But Is It User Friendly?</td>
      <td>popular_comp</td>
      <td>1389</td>
      <td>22.9500</td>
      <td>7000.0000</td>
      <td>16</td>
      <td>8780</td>
      <td>A survey of software for the naive user, focusing on the 'friendliness' of each.</td>
      <td>1991-06-30 00:00:00.000</td>
    </tr>
    <tr>
      <td>PC8888</td>
      <td>Secrets of Silicon Valley</td>
      <td>popular_comp</td>
      <td>1389</td>
      <td>20.0000</td>
      <td>8000.0000</td>
      <td>10</td>
      <td>4095</td>
      <td>Muckraking reporting on the world's largest computer hardware and software manufacturers.</td>
      <td>1994-06-12 00:00:00.000</td>
    </tr>
    <tr>
      <td>PC9999</td>
      <td>Net Etiquette</td>
      <td>popular_comp</td>
      <td>1389</td>
      <td>NULL</td>
      <td>NULL</td>
      <td>NULL</td>
      <td>NULL</td>
      <td>A must-read for computer conferencing.</td>
      <td>2026-10-03 00:00:00.000</td>
    </tr>
    <tr>
      <td>PS1372</td>
      <td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td>
      <td>psychology</td>
      <td>0877</td>
      <td>21.5900</td>
      <td>7000.0000</td>
      <td>10</td>
      <td>375</td>
      <td>A must for the specialist, this book examines the difference between those who hate and fear computers and those who don't.</td>
      <td>1991-10-21 00:00:00.000</td>
    </tr>
    <tr>
      <td>PS2091</td>
      <td>Is Anger the Enemy?</td>
      <td>psychology</td>
      <td>0736</td>
      <td>10.9500</td>
      <td>2275.0000</td>
      <td>12</td>
      <td>2045</td>
      <td>Carefully researched study of the effects of strong emotions on the body. Metabolic charts included.</td>
      <td>1991-06-15 00:00:00.000</td>
    </tr>
    <tr>
      <td>PS2106</td>
      <td>Life Without Fear</td>
      <td>psychology</td>
      <td>0736</td>
      <td>7.0000</td>
      <td>6000.0000</td>
      <td>10</td>
      <td>111</td>
      <td>New exercise, meditation, and nutritional techniques that can reduce the shock of daily interactions. Popular audience. Sample menus included, exercise video available separately.</td>
      <td>1991-10-05 00:00:00.000</td>
    </tr>
    <tr>
      <td>PS3333</td>
      <td>Prolonged Data Deprivation: Four Case Studies</td>
      <td>psychology</td>
      <td>0736</td>
      <td>19.9900</td>
      <td>2000.0000</td>
      <td>10</td>
      <td>4072</td>
      <td>What happens when the data runs dry?&nbsp;&nbsp;Searching evaluations of information-shortage effects.</td>
      <td>1991-06-12 00:00:00.000</td>
    </tr>
    <tr>
      <td>PS7777</td>
      <td>Emotional Security: A New Algorithm</td>
      <td>psychology</td>
      <td>0736</td>
      <td>7.9900</td>
      <td>4000.0000</td>
      <td>10</td>
      <td>3336</td>
      <td>Protecting yourself and your loved ones from undue emotional stress in the modern world. Use of computer and nutritional aids emphasized.</td>
      <td>1991-06-12 00:00:00.000</td>
    </tr>
    <tr>
      <td>TC3218</td>
      <td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td>
      <td>trad_cook</td>
      <td>0877</td>
      <td>20.9500</td>
      <td>7000.0000</td>
      <td>10</td>
      <td>375</td>
      <td>Profusely illustrated in color, this makes a wonderful gift book for a cuisine-oriented friend.</td>
      <td>1991-10-21 00:00:00.000</td>
    </tr>
    <tr>
      <td>TC4203</td>
      <td>Fifty Years in Buckingham Palace Kitchens</td>
      <td>trad_cook</td>
      <td>0877</td>
      <td>11.9500</td>
      <td>4000.0000</td>
      <td>14</td>
      <td>15096</td>
      <td>More anecdotes from the Queen's favorite cook describing life among English royalty. Recipes, techniques, tender vignettes.</td>
      <td>1991-06-12 00:00:00.000</td>
    </tr>
    <tr>
      <td>TC7777</td>
      <td>Sushi, Anyone?</td>
      <td>trad_cook</td>
      <td>0877</td>
      <td>14.9900</td>
      <td>8000.0000</td>
      <td>10</td>
      <td>4095</td>
      <td>Detailed instructions on how to make authentic Japanese sushi in your spare time.</td>
      <td>1991-06-12 00:00:00.000</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">18 row(s) — order not guaranteed without ORDER BY (shown in clustered-index order)</p></div>

**Example 1.2 — choose columns, give the table an alias.** Writing `t.title` instead of `title` is the habit the instructor uses throughout: the alias `t` makes every column's origin explicit, which becomes essential once two tables are involved.

In [ ]:
SELECT t.title_id, t.title, t.price
FROM dbo.titles AS t;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title_id</th>
      <th>title</th>
      <th>price</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>BU1032</td>
      <td>The Busy Executive's Database Guide</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>BU1111</td>
      <td>Cooking with Computers: Surreptitious Balance Sheets</td>
      <td>11.9500</td>
    </tr>
    <tr>
      <td>BU2075</td>
      <td>You Can Combat Computer Stress!</td>
      <td>2.9900</td>
    </tr>
    <tr>
      <td>BU7832</td>
      <td>Straight Talk About Computers</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>MC2222</td>
      <td>Silicon Valley Gastronomic Treats</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>MC3021</td>
      <td>The Gourmet Microwave</td>
      <td>2.9900</td>
    </tr>
    <tr>
      <td>MC3026</td>
      <td>The Psychology of Computer Cooking</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>PC1035</td>
      <td>But Is It User Friendly?</td>
      <td>22.9500</td>
    </tr>
    <tr>
      <td>PC8888</td>
      <td>Secrets of Silicon Valley</td>
      <td>20.0000</td>
    </tr>
    <tr>
      <td>PC9999</td>
      <td>Net Etiquette</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>PS1372</td>
      <td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td>
      <td>21.5900</td>
    </tr>
    <tr>
      <td>PS2091</td>
      <td>Is Anger the Enemy?</td>
      <td>10.9500</td>
    </tr>
    <tr>
      <td>PS2106</td>
      <td>Life Without Fear</td>
      <td>7.0000</td>
    </tr>
    <tr>
      <td>PS3333</td>
      <td>Prolonged Data Deprivation: Four Case Studies</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>PS7777</td>
      <td>Emotional Security: A New Algorithm</td>
      <td>7.9900</td>
    </tr>
    <tr>
      <td>TC3218</td>
      <td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td>
      <td>20.9500</td>
    </tr>
    <tr>
      <td>TC4203</td>
      <td>Fifty Years in Buckingham Palace Kitchens</td>
      <td>11.9500</td>
    </tr>
    <tr>
      <td>TC7777</td>
      <td>Sushi, Anyone?</td>
      <td>14.9900</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">18 row(s) — order not guaranteed without ORDER BY</p></div>

**Example 1.3 — ascending is the default.** The first query has no direction; the second spells out `ASC`. They return identical results. Look at where the `NULL` prices appear: at the top.

In [ ]:
SELECT t.title_id, t.title, t.price
FROM dbo.titles AS t
ORDER BY t.price;

SELECT t.title_id, t.title, t.price
FROM dbo.titles AS t
ORDER BY t.price ASC;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title_id</th>
      <th>title</th>
      <th>price</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>MC3026</td>
      <td>The Psychology of Computer Cooking</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>PC9999</td>
      <td>Net Etiquette</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>BU2075</td>
      <td>You Can Combat Computer Stress!</td>
      <td>2.9900</td>
    </tr>
    <tr>
      <td>MC3021</td>
      <td>The Gourmet Microwave</td>
      <td>2.9900</td>
    </tr>
    <tr>
      <td>PS2106</td>
      <td>Life Without Fear</td>
      <td>7.0000</td>
    </tr>
    <tr>
      <td>PS7777</td>
      <td>Emotional Security: A New Algorithm</td>
      <td>7.9900</td>
    </tr>
    <tr>
      <td>PS2091</td>
      <td>Is Anger the Enemy?</td>
      <td>10.9500</td>
    </tr>
    <tr>
      <td>BU1111</td>
      <td>Cooking with Computers: Surreptitious Balance Sheets</td>
      <td>11.9500</td>
    </tr>
    <tr>
      <td>TC4203</td>
      <td>Fifty Years in Buckingham Palace Kitchens</td>
      <td>11.9500</td>
    </tr>
    <tr>
      <td>TC7777</td>
      <td>Sushi, Anyone?</td>
      <td>14.9900</td>
    </tr>
    <tr>
      <td>BU1032</td>
      <td>The Busy Executive's Database Guide</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>BU7832</td>
      <td>Straight Talk About Computers</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>MC2222</td>
      <td>Silicon Valley Gastronomic Treats</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>PS3333</td>
      <td>Prolonged Data Deprivation: Four Case Studies</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>PC8888</td>
      <td>Secrets of Silicon Valley</td>
      <td>20.0000</td>
    </tr>
    <tr>
      <td>TC3218</td>
      <td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td>
      <td>20.9500</td>
    </tr>
    <tr>
      <td>PS1372</td>
      <td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td>
      <td>21.5900</td>
    </tr>
    <tr>
      <td>PC1035</td>
      <td>But Is It User Friendly?</td>
      <td>22.9500</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">18 row(s) — order among equal prices is not guaranteed</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title_id</th>
      <th>title</th>
      <th>price</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>MC3026</td>
      <td>The Psychology of Computer Cooking</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>PC9999</td>
      <td>Net Etiquette</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>BU2075</td>
      <td>You Can Combat Computer Stress!</td>
      <td>2.9900</td>
    </tr>
    <tr>
      <td>MC3021</td>
      <td>The Gourmet Microwave</td>
      <td>2.9900</td>
    </tr>
    <tr>
      <td>PS2106</td>
      <td>Life Without Fear</td>
      <td>7.0000</td>
    </tr>
    <tr>
      <td>PS7777</td>
      <td>Emotional Security: A New Algorithm</td>
      <td>7.9900</td>
    </tr>
    <tr>
      <td>PS2091</td>
      <td>Is Anger the Enemy?</td>
      <td>10.9500</td>
    </tr>
    <tr>
      <td>BU1111</td>
      <td>Cooking with Computers: Surreptitious Balance Sheets</td>
      <td>11.9500</td>
    </tr>
    <tr>
      <td>TC4203</td>
      <td>Fifty Years in Buckingham Palace Kitchens</td>
      <td>11.9500</td>
    </tr>
    <tr>
      <td>TC7777</td>
      <td>Sushi, Anyone?</td>
      <td>14.9900</td>
    </tr>
    <tr>
      <td>BU1032</td>
      <td>The Busy Executive's Database Guide</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>BU7832</td>
      <td>Straight Talk About Computers</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>MC2222</td>
      <td>Silicon Valley Gastronomic Treats</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>PS3333</td>
      <td>Prolonged Data Deprivation: Four Case Studies</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>PC8888</td>
      <td>Secrets of Silicon Valley</td>
      <td>20.0000</td>
    </tr>
    <tr>
      <td>TC3218</td>
      <td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td>
      <td>20.9500</td>
    </tr>
    <tr>
      <td>PS1372</td>
      <td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td>
      <td>21.5900</td>
    </tr>
    <tr>
      <td>PC1035</td>
      <td>But Is It User Friendly?</td>
      <td>22.9500</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">18 row(s) — identical to the first result</p></div>

**Example 1.4 — descending.** Now the most expensive books come first and the `NULL` prices sink to the bottom.

In [ ]:
SELECT t.title_id, t.title, t.price
FROM dbo.titles AS t
ORDER BY t.price DESC;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title_id</th>
      <th>title</th>
      <th>price</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>PC1035</td>
      <td>But Is It User Friendly?</td>
      <td>22.9500</td>
    </tr>
    <tr>
      <td>PS1372</td>
      <td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td>
      <td>21.5900</td>
    </tr>
    <tr>
      <td>TC3218</td>
      <td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td>
      <td>20.9500</td>
    </tr>
    <tr>
      <td>PC8888</td>
      <td>Secrets of Silicon Valley</td>
      <td>20.0000</td>
    </tr>
    <tr>
      <td>BU1032</td>
      <td>The Busy Executive's Database Guide</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>BU7832</td>
      <td>Straight Talk About Computers</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>MC2222</td>
      <td>Silicon Valley Gastronomic Treats</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>PS3333</td>
      <td>Prolonged Data Deprivation: Four Case Studies</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>TC7777</td>
      <td>Sushi, Anyone?</td>
      <td>14.9900</td>
    </tr>
    <tr>
      <td>BU1111</td>
      <td>Cooking with Computers: Surreptitious Balance Sheets</td>
      <td>11.9500</td>
    </tr>
    <tr>
      <td>TC4203</td>
      <td>Fifty Years in Buckingham Palace Kitchens</td>
      <td>11.9500</td>
    </tr>
    <tr>
      <td>PS2091</td>
      <td>Is Anger the Enemy?</td>
      <td>10.9500</td>
    </tr>
    <tr>
      <td>PS7777</td>
      <td>Emotional Security: A New Algorithm</td>
      <td>7.9900</td>
    </tr>
    <tr>
      <td>PS2106</td>
      <td>Life Without Fear</td>
      <td>7.0000</td>
    </tr>
    <tr>
      <td>BU2075</td>
      <td>You Can Combat Computer Stress!</td>
      <td>2.9900</td>
    </tr>
    <tr>
      <td>MC3021</td>
      <td>The Gourmet Microwave</td>
      <td>2.9900</td>
    </tr>
    <tr>
      <td>MC3026</td>
      <td>The Psychology of Computer Cooking</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>PC9999</td>
      <td>Net Etiquette</td>
      <td>NULL</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">18 row(s) — order among equal prices is not guaranteed</p></div>

**Example 1.5 — replacing `NULL` with `ISNULL` and sorting by the alias.**
`ISNULL(expression, replacement)` substitutes a value when the expression is `NULL`. Here a missing price is shown as `0`. The first query sorts by the original column; the second sorts by the alias `BookPrice`, which is legal because `ORDER BY` is processed after `SELECT`. With the alias in ascending order, the unpriced books (now showing `0`) come first.

In [ ]:
SELECT t.title_id, t.title, ISNULL(t.price, 0) AS BookPrice
FROM dbo.titles AS t
ORDER BY t.price DESC;

SELECT t.title_id, t.title, ISNULL(t.price, 0) AS BookPrice
FROM dbo.titles AS t
ORDER BY BookPrice ASC;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title_id</th>
      <th>title</th>
      <th>BookPrice</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>PC1035</td>
      <td>But Is It User Friendly?</td>
      <td>22.9500</td>
    </tr>
    <tr>
      <td>PS1372</td>
      <td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td>
      <td>21.5900</td>
    </tr>
    <tr>
      <td>TC3218</td>
      <td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td>
      <td>20.9500</td>
    </tr>
    <tr>
      <td>PC8888</td>
      <td>Secrets of Silicon Valley</td>
      <td>20.0000</td>
    </tr>
    <tr>
      <td>BU1032</td>
      <td>The Busy Executive's Database Guide</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>BU7832</td>
      <td>Straight Talk About Computers</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>MC2222</td>
      <td>Silicon Valley Gastronomic Treats</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>PS3333</td>
      <td>Prolonged Data Deprivation: Four Case Studies</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>TC7777</td>
      <td>Sushi, Anyone?</td>
      <td>14.9900</td>
    </tr>
    <tr>
      <td>BU1111</td>
      <td>Cooking with Computers: Surreptitious Balance Sheets</td>
      <td>11.9500</td>
    </tr>
    <tr>
      <td>TC4203</td>
      <td>Fifty Years in Buckingham Palace Kitchens</td>
      <td>11.9500</td>
    </tr>
    <tr>
      <td>PS2091</td>
      <td>Is Anger the Enemy?</td>
      <td>10.9500</td>
    </tr>
    <tr>
      <td>PS7777</td>
      <td>Emotional Security: A New Algorithm</td>
      <td>7.9900</td>
    </tr>
    <tr>
      <td>PS2106</td>
      <td>Life Without Fear</td>
      <td>7.0000</td>
    </tr>
    <tr>
      <td>BU2075</td>
      <td>You Can Combat Computer Stress!</td>
      <td>2.9900</td>
    </tr>
    <tr>
      <td>MC3021</td>
      <td>The Gourmet Microwave</td>
      <td>2.9900</td>
    </tr>
    <tr>
      <td>MC3026</td>
      <td>The Psychology of Computer Cooking</td>
      <td>0.0000</td>
    </tr>
    <tr>
      <td>PC9999</td>
      <td>Net Etiquette</td>
      <td>0.0000</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">18 row(s)</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title_id</th>
      <th>title</th>
      <th>BookPrice</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>MC3026</td>
      <td>The Psychology of Computer Cooking</td>
      <td>0.0000</td>
    </tr>
    <tr>
      <td>PC9999</td>
      <td>Net Etiquette</td>
      <td>0.0000</td>
    </tr>
    <tr>
      <td>BU2075</td>
      <td>You Can Combat Computer Stress!</td>
      <td>2.9900</td>
    </tr>
    <tr>
      <td>MC3021</td>
      <td>The Gourmet Microwave</td>
      <td>2.9900</td>
    </tr>
    <tr>
      <td>PS2106</td>
      <td>Life Without Fear</td>
      <td>7.0000</td>
    </tr>
    <tr>
      <td>PS7777</td>
      <td>Emotional Security: A New Algorithm</td>
      <td>7.9900</td>
    </tr>
    <tr>
      <td>PS2091</td>
      <td>Is Anger the Enemy?</td>
      <td>10.9500</td>
    </tr>
    <tr>
      <td>BU1111</td>
      <td>Cooking with Computers: Surreptitious Balance Sheets</td>
      <td>11.9500</td>
    </tr>
    <tr>
      <td>TC4203</td>
      <td>Fifty Years in Buckingham Palace Kitchens</td>
      <td>11.9500</td>
    </tr>
    <tr>
      <td>TC7777</td>
      <td>Sushi, Anyone?</td>
      <td>14.9900</td>
    </tr>
    <tr>
      <td>BU1032</td>
      <td>The Busy Executive's Database Guide</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>BU7832</td>
      <td>Straight Talk About Computers</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>MC2222</td>
      <td>Silicon Valley Gastronomic Treats</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>PS3333</td>
      <td>Prolonged Data Deprivation: Four Case Studies</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>PC8888</td>
      <td>Secrets of Silicon Valley</td>
      <td>20.0000</td>
    </tr>
    <tr>
      <td>TC3218</td>
      <td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td>
      <td>20.9500</td>
    </tr>
    <tr>
      <td>PS1372</td>
      <td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td>
      <td>21.5900</td>
    </tr>
    <tr>
      <td>PC1035</td>
      <td>But Is It User Friendly?</td>
      <td>22.9500</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">18 row(s)</p></div>

**Example 1.6 — sorting by column position.** `ORDER BY 3` means "the third column in the `SELECT` list" (here `BookPrice`). It works, but if someone adds or reorders a column the sort silently changes. This is what the instructor's comment *Self-Explainer* is pointing at: write code that explains itself, and prefer names over numbers.

In [ ]:
-- Works, but fragile: "3" means "the third column in the SELECT list"
SELECT t.title_id, t.title, ISNULL(t.price, 0) AS BookPrice
FROM dbo.titles AS t
ORDER BY 3 DESC;

-- Self-explaining version
SELECT t.title_id, t.title, ISNULL(t.price, 0) AS BookPrice
FROM dbo.titles AS t
ORDER BY BookPrice DESC;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title_id</th>
      <th>title</th>
      <th>BookPrice</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>PC1035</td>
      <td>But Is It User Friendly?</td>
      <td>22.9500</td>
    </tr>
    <tr>
      <td>PS1372</td>
      <td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td>
      <td>21.5900</td>
    </tr>
    <tr>
      <td>TC3218</td>
      <td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td>
      <td>20.9500</td>
    </tr>
    <tr>
      <td>PC8888</td>
      <td>Secrets of Silicon Valley</td>
      <td>20.0000</td>
    </tr>
    <tr>
      <td>BU1032</td>
      <td>The Busy Executive's Database Guide</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>BU7832</td>
      <td>Straight Talk About Computers</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>MC2222</td>
      <td>Silicon Valley Gastronomic Treats</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>PS3333</td>
      <td>Prolonged Data Deprivation: Four Case Studies</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>TC7777</td>
      <td>Sushi, Anyone?</td>
      <td>14.9900</td>
    </tr>
    <tr>
      <td>BU1111</td>
      <td>Cooking with Computers: Surreptitious Balance Sheets</td>
      <td>11.9500</td>
    </tr>
    <tr>
      <td>TC4203</td>
      <td>Fifty Years in Buckingham Palace Kitchens</td>
      <td>11.9500</td>
    </tr>
    <tr>
      <td>PS2091</td>
      <td>Is Anger the Enemy?</td>
      <td>10.9500</td>
    </tr>
    <tr>
      <td>PS7777</td>
      <td>Emotional Security: A New Algorithm</td>
      <td>7.9900</td>
    </tr>
    <tr>
      <td>PS2106</td>
      <td>Life Without Fear</td>
      <td>7.0000</td>
    </tr>
    <tr>
      <td>BU2075</td>
      <td>You Can Combat Computer Stress!</td>
      <td>2.9900</td>
    </tr>
    <tr>
      <td>MC3021</td>
      <td>The Gourmet Microwave</td>
      <td>2.9900</td>
    </tr>
    <tr>
      <td>MC3026</td>
      <td>The Psychology of Computer Cooking</td>
      <td>0.0000</td>
    </tr>
    <tr>
      <td>PC9999</td>
      <td>Net Etiquette</td>
      <td>0.0000</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">18 row(s) — same as the next result</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title_id</th>
      <th>title</th>
      <th>BookPrice</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>PC1035</td>
      <td>But Is It User Friendly?</td>
      <td>22.9500</td>
    </tr>
    <tr>
      <td>PS1372</td>
      <td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td>
      <td>21.5900</td>
    </tr>
    <tr>
      <td>TC3218</td>
      <td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td>
      <td>20.9500</td>
    </tr>
    <tr>
      <td>PC8888</td>
      <td>Secrets of Silicon Valley</td>
      <td>20.0000</td>
    </tr>
    <tr>
      <td>BU1032</td>
      <td>The Busy Executive's Database Guide</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>BU7832</td>
      <td>Straight Talk About Computers</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>MC2222</td>
      <td>Silicon Valley Gastronomic Treats</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>PS3333</td>
      <td>Prolonged Data Deprivation: Four Case Studies</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>TC7777</td>
      <td>Sushi, Anyone?</td>
      <td>14.9900</td>
    </tr>
    <tr>
      <td>BU1111</td>
      <td>Cooking with Computers: Surreptitious Balance Sheets</td>
      <td>11.9500</td>
    </tr>
    <tr>
      <td>TC4203</td>
      <td>Fifty Years in Buckingham Palace Kitchens</td>
      <td>11.9500</td>
    </tr>
    <tr>
      <td>PS2091</td>
      <td>Is Anger the Enemy?</td>
      <td>10.9500</td>
    </tr>
    <tr>
      <td>PS7777</td>
      <td>Emotional Security: A New Algorithm</td>
      <td>7.9900</td>
    </tr>
    <tr>
      <td>PS2106</td>
      <td>Life Without Fear</td>
      <td>7.0000</td>
    </tr>
    <tr>
      <td>BU2075</td>
      <td>You Can Combat Computer Stress!</td>
      <td>2.9900</td>
    </tr>
    <tr>
      <td>MC3021</td>
      <td>The Gourmet Microwave</td>
      <td>2.9900</td>
    </tr>
    <tr>
      <td>MC3026</td>
      <td>The Psychology of Computer Cooking</td>
      <td>0.0000</td>
    </tr>
    <tr>
      <td>PC9999</td>
      <td>Net Etiquette</td>
      <td>0.0000</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">18 row(s)</p></div>

**Example 1.7 — several sort columns (tie-breaking).** Several books share the same price. With `ORDER BY price DESC` alone, the order *among* equally priced books is arbitrary. Adding `t.title ASC` makes the result fully deterministic: sort by price first, and within equal prices sort alphabetically.

In [ ]:
SELECT t.title_id, t.title, ISNULL(t.price, 0) AS BookPrice
FROM dbo.titles AS t
ORDER BY t.price DESC, t.title ASC;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title_id</th>
      <th>title</th>
      <th>BookPrice</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>PC1035</td>
      <td>But Is It User Friendly?</td>
      <td>22.9500</td>
    </tr>
    <tr>
      <td>PS1372</td>
      <td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td>
      <td>21.5900</td>
    </tr>
    <tr>
      <td>TC3218</td>
      <td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td>
      <td>20.9500</td>
    </tr>
    <tr>
      <td>PC8888</td>
      <td>Secrets of Silicon Valley</td>
      <td>20.0000</td>
    </tr>
    <tr>
      <td>PS3333</td>
      <td>Prolonged Data Deprivation: Four Case Studies</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>MC2222</td>
      <td>Silicon Valley Gastronomic Treats</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>BU7832</td>
      <td>Straight Talk About Computers</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>BU1032</td>
      <td>The Busy Executive's Database Guide</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>TC7777</td>
      <td>Sushi, Anyone?</td>
      <td>14.9900</td>
    </tr>
    <tr>
      <td>BU1111</td>
      <td>Cooking with Computers: Surreptitious Balance Sheets</td>
      <td>11.9500</td>
    </tr>
    <tr>
      <td>TC4203</td>
      <td>Fifty Years in Buckingham Palace Kitchens</td>
      <td>11.9500</td>
    </tr>
    <tr>
      <td>PS2091</td>
      <td>Is Anger the Enemy?</td>
      <td>10.9500</td>
    </tr>
    <tr>
      <td>PS7777</td>
      <td>Emotional Security: A New Algorithm</td>
      <td>7.9900</td>
    </tr>
    <tr>
      <td>PS2106</td>
      <td>Life Without Fear</td>
      <td>7.0000</td>
    </tr>
    <tr>
      <td>MC3021</td>
      <td>The Gourmet Microwave</td>
      <td>2.9900</td>
    </tr>
    <tr>
      <td>BU2075</td>
      <td>You Can Combat Computer Stress!</td>
      <td>2.9900</td>
    </tr>
    <tr>
      <td>PC9999</td>
      <td>Net Etiquette</td>
      <td>0.0000</td>
    </tr>
    <tr>
      <td>MC3026</td>
      <td>The Psychology of Computer Cooking</td>
      <td>0.0000</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">18 row(s)</p></div>

**Example 1.8 — sorting by an expression, or by a column you do not display.** `ORDER BY` accepts any expression. The first query sorts by the length of the title; the second sorts by price without showing price.

مثال ۱٫۸ دو نکته‌ی جدا دارد، و هر دو یک ایده‌ی مشترک دارند: چیزی که جلوی `ORDER BY` می‌نویسی لازم نیست حتماً اسم یک ستونِ نمایش‌داده‌شده باشد.

**نکته‌ی اول: مرتب‌سازی بر اساس یک عبارت محاسبه‌شده**

```sql
SELECT t.title, LEN(t.title) AS TitleLength
FROM dbo.titles AS t
ORDER BY LEN(t.title) DESC;
```

`LEN(t.title)` تعداد کاراکترهای عنوان را برمی‌گرداند. مثلاً عنوان «Net Etiquette» سیزده کاراکتر دارد. SQL Server این عدد را برای هر ردیف حساب می‌کند و ردیف‌ها را بر اساس همین عدد مرتب می‌کند، نه بر اساس خودِ متن عنوان. چون `DESC` گذاشته‌ایم، طولانی‌ترین عنوان‌ها اول می‌آیند.

پس `ORDER BY` فقط برای ستون‌های خام نیست. هر چیزی که برای هر ردیف یک مقدار بدهد قابل مرتب‌سازی است، مثل `LEN(...)`، `price * 2` یا `YEAR(pubdate)`.

**نکته‌ی دوم: مرتب‌سازی بر اساس ستونی که نشانش نمی‌دهی**

```sql
SELECT t.title
FROM dbo.titles AS t
ORDER BY t.price DESC;
```

در `SELECT` فقط `title` را خواسته‌ایم، ولی ترتیب نمایش بر اساس `price` است. خروجی فقط ستون عنوان دارد، اما اولین عنوان، عنوانِ گران‌ترین کتاب است و آخرین، ارزان‌ترین. این مثل این است که دانش‌آموزان را بر اساس قدشان در صف بایستانی، ولی قد هیچ‌کس را روی تابلو ننویسی.

کاربردش وقتی است که فقط فهرست اسامی را می‌خواهی، ولی ترتیبش باید بر اساس معیاری باشد که لازم نیست دیده شود. اشکالش این است که از روی خروجی نمی‌توانی درستی ترتیب را چک کنی. اگر می‌خواهی ببینی، موقتاً `t.price` را به `SELECT` اضافه کن.

یک محدودیت هم دارد که در مثال ۱٫۹ آمده: اگر در `SELECT` از `DISTINCT` استفاده کنی، دیگر نمی‌توانی بر اساس ستونی که در `SELECT` نیست مرتب کنی و SQL Server خطا می‌دهد.

اگر بخواهی، همین دو query را روی یک مثال کوچک‌تر، مثلاً سه کتاب، قدم‌به‌قدم می‌نویسم تا خودت ببینی هر ردیف چه عددی می‌گیرد و چرا آن ترتیب درمی‌آید.

In [ ]:
SELECT t.title, LEN(t.title) AS TitleLength
FROM dbo.titles AS t
ORDER BY LEN(t.title) DESC;

SELECT t.title
FROM dbo.titles AS t
ORDER BY t.price DESC;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title</th>
      <th>TitleLength</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td>
      <td>63</td>
    </tr>
    <tr>
      <td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td>
      <td>63</td>
    </tr>
    <tr>
      <td>Cooking with Computers: Surreptitious Balance Sheets</td>
      <td>52</td>
    </tr>
    <tr>
      <td>Prolonged Data Deprivation: Four Case Studies</td>
      <td>45</td>
    </tr>
    <tr>
      <td>Fifty Years in Buckingham Palace Kitchens</td>
      <td>41</td>
    </tr>
    <tr>
      <td>The Busy Executive's Database Guide</td>
      <td>35</td>
    </tr>
    <tr>
      <td>Emotional Security: A New Algorithm</td>
      <td>35</td>
    </tr>
    <tr>
      <td>The Psychology of Computer Cooking</td>
      <td>34</td>
    </tr>
    <tr>
      <td>Silicon Valley Gastronomic Treats</td>
      <td>33</td>
    </tr>
    <tr>
      <td>You Can Combat Computer Stress!</td>
      <td>31</td>
    </tr>
    <tr>
      <td>Straight Talk About Computers</td>
      <td>29</td>
    </tr>
    <tr>
      <td>Secrets of Silicon Valley</td>
      <td>25</td>
    </tr>
    <tr>
      <td>But Is It User Friendly?</td>
      <td>24</td>
    </tr>
    <tr>
      <td>The Gourmet Microwave</td>
      <td>21</td>
    </tr>
    <tr>
      <td>Is Anger the Enemy?</td>
      <td>19</td>
    </tr>
    <tr>
      <td>Life Without Fear</td>
      <td>17</td>
    </tr>
    <tr>
      <td>Sushi, Anyone?</td>
      <td>14</td>
    </tr>
    <tr>
      <td>Net Etiquette</td>
      <td>13</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">18 row(s)</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>But Is It User Friendly?</td>
    </tr>
    <tr>
      <td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td>
    </tr>
    <tr>
      <td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td>
    </tr>
    <tr>
      <td>Secrets of Silicon Valley</td>
    </tr>
    <tr>
      <td>The Busy Executive's Database Guide</td>
    </tr>
    <tr>
      <td>Straight Talk About Computers</td>
    </tr>
    <tr>
      <td>Silicon Valley Gastronomic Treats</td>
    </tr>
    <tr>
      <td>Prolonged Data Deprivation: Four Case Studies</td>
    </tr>
    <tr>
      <td>Sushi, Anyone?</td>
    </tr>
    <tr>
      <td>Cooking with Computers: Surreptitious Balance Sheets</td>
    </tr>
    <tr>
      <td>Fifty Years in Buckingham Palace Kitchens</td>
    </tr>
    <tr>
      <td>Is Anger the Enemy?</td>
    </tr>
    <tr>
      <td>Emotional Security: A New Algorithm</td>
    </tr>
    <tr>
      <td>Life Without Fear</td>
    </tr>
    <tr>
      <td>You Can Combat Computer Stress!</td>
    </tr>
    <tr>
      <td>The Gourmet Microwave</td>
    </tr>
    <tr>
      <td>The Psychology of Computer Cooking</td>
    </tr>
    <tr>
      <td>Net Etiquette</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">18 row(s) — titles with equal prices may appear in any order</p></div>

**Example 1.9 — a restriction worth knowing.** With `SELECT DISTINCT`, every `ORDER BY` item must appear in the `SELECT` list. SQL Server cannot sort by a value that the de-duplication step has just thrown away.

In [ ]:
SELECT DISTINCT t.[type]
FROM dbo.titles AS t
ORDER BY t.price;

Msg 145, Level 15, State 1, Line 1
ORDER BY items must appear in the select list if SELECT DISTINCT is specified.

In [ ]:
-- Fixed: sort by something that is in the select list
SELECT DISTINCT t.[type]
FROM dbo.titles AS t
ORDER BY t.[type];

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>type</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>business</td>
    </tr>
    <tr>
      <td>mod_cook</td>
    </tr>
    <tr>
      <td>popular_comp</td>
    </tr>
    <tr>
      <td>psychology</td>
    </tr>
    <tr>
      <td>trad_cook</td>
    </tr>
    <tr>
      <td>UNDECIDED</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">6 row(s)</p></div>

### Pitfalls — `ORDER BY`
- No `ORDER BY` means no guaranteed order, even if the output looks sorted today.
- `ORDER BY` is not allowed in views, derived tables or subqueries unless `TOP` (or `OFFSET`) is also used, because a table expression is a set.
- Ordinal positions (`ORDER BY 3`) break silently when the `SELECT` list changes.
- Remember that `NULL` sorts first in ASC and last in DESC; use `ISNULL`/`COALESCE` or a `CASE` expression if you need different behaviour.

---
## 2. `TOP N` — limiting the number of rows

### Purpose
`TOP` returns only the first *n* rows (or the first *p* percent of rows) of a result. Typical uses: "the most expensive book", "the 10 best customers", a quick peek at a huge table, and (combined with `NEWID()`) a random sample.

### Syntax
```sql
SELECT [DISTINCT] TOP (n) [PERCENT] [WITH TIES]
       column_list
FROM   ...
[ORDER BY ...]
```

### Key rules
- `TOP` only makes sense **together with `ORDER BY`**. Without it, you get *some* n rows, and you cannot say which.
- `WITH TIES` **requires** `ORDER BY`: ties are defined by the sort column.
- `PERCENT` rounds *up*: 30 percent of 18 rows is 5.4, so you get 6 rows.
- Parentheses, `TOP (5)`, are the recommended form and are required when n is an expression or a variable.
- `TOP` is T-SQL specific. The ANSI-standard equivalent is `OFFSET ... FETCH` (Example 2.7).

**Example 2.1 — `TOP 5` with no `ORDER BY`.** You get five rows, but they are simply the first five SQL Server happened to read. Treat this as "give me a sample", not "give me the first five".

In [ ]:
SELECT TOP 5
       t.title,
       t.[type],
       t.price,
       t.pubdate
FROM dbo.titles AS t;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title</th>
      <th>type</th>
      <th>price</th>
      <th>pubdate</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>The Busy Executive's Database Guide</td>
      <td>business</td>
      <td>19.9900</td>
      <td>1991-06-12 00:00:00.000</td>
    </tr>
    <tr>
      <td>Cooking with Computers: Surreptitious Balance Sheets</td>
      <td>business</td>
      <td>11.9500</td>
      <td>1991-06-09 00:00:00.000</td>
    </tr>
    <tr>
      <td>You Can Combat Computer Stress!</td>
      <td>business</td>
      <td>2.9900</td>
      <td>1991-06-30 00:00:00.000</td>
    </tr>
    <tr>
      <td>Straight Talk About Computers</td>
      <td>business</td>
      <td>19.9900</td>
      <td>1991-06-22 00:00:00.000</td>
    </tr>
    <tr>
      <td>Silicon Valley Gastronomic Treats</td>
      <td>mod_cook</td>
      <td>19.9900</td>
      <td>1991-06-09 00:00:00.000</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">5 row(s) — which five rows you get is not guaranteed</p></div>

**Example 2.2 — `TOP 1` with `ORDER BY ... DESC`: the single most expensive book.** This is the classic pattern: sort so that the row you want comes first, then take one.

In [ ]:
SELECT TOP 1
       t.title,
       t.[type],
       t.price,
       t.pubdate
FROM dbo.titles AS t
ORDER BY t.price DESC;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title</th>
      <th>type</th>
      <th>price</th>
      <th>pubdate</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>But Is It User Friendly?</td>
      <td>popular_comp</td>
      <td>22.9500</td>
      <td>1991-06-30 00:00:00.000</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">1 row(s)</p></div>

**Example 2.3 — `WITH TIES`.** Plain `TOP 5` cuts after exactly five rows, even if the fifth and sixth books cost the same, which makes the cut unfair and arbitrary. `WITH TIES` keeps extending the result as long as the following rows tie with the last row on the `ORDER BY` value. Compare the row counts of the two queries. Several books in `pubs` share the price 19.99, so the second query returns **8 rows instead of 5**.

In [ ]:
SELECT TOP 5
       t.title, t.[type], t.price, t.pubdate
FROM dbo.titles AS t
ORDER BY t.price DESC;

SELECT TOP 5 WITH TIES
       t.title, t.[type], t.price, t.pubdate
FROM dbo.titles AS t
ORDER BY t.price DESC;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title</th>
      <th>type</th>
      <th>price</th>
      <th>pubdate</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>But Is It User Friendly?</td>
      <td>popular_comp</td>
      <td>22.9500</td>
      <td>1991-06-30 00:00:00.000</td>
    </tr>
    <tr>
      <td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td>
      <td>psychology</td>
      <td>21.5900</td>
      <td>1991-10-21 00:00:00.000</td>
    </tr>
    <tr>
      <td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td>
      <td>trad_cook</td>
      <td>20.9500</td>
      <td>1991-10-21 00:00:00.000</td>
    </tr>
    <tr>
      <td>Secrets of Silicon Valley</td>
      <td>popular_comp</td>
      <td>20.0000</td>
      <td>1994-06-12 00:00:00.000</td>
    </tr>
    <tr>
      <td>The Busy Executive's Database Guide</td>
      <td>business</td>
      <td>19.9900</td>
      <td>1991-06-12 00:00:00.000</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">5 row(s) — the fifth row is one of four books priced 19.9900; which one is arbitrary</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title</th>
      <th>type</th>
      <th>price</th>
      <th>pubdate</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>But Is It User Friendly?</td>
      <td>popular_comp</td>
      <td>22.9500</td>
      <td>1991-06-30 00:00:00.000</td>
    </tr>
    <tr>
      <td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td>
      <td>psychology</td>
      <td>21.5900</td>
      <td>1991-10-21 00:00:00.000</td>
    </tr>
    <tr>
      <td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td>
      <td>trad_cook</td>
      <td>20.9500</td>
      <td>1991-10-21 00:00:00.000</td>
    </tr>
    <tr>
      <td>Secrets of Silicon Valley</td>
      <td>popular_comp</td>
      <td>20.0000</td>
      <td>1994-06-12 00:00:00.000</td>
    </tr>
    <tr>
      <td>The Busy Executive's Database Guide</td>
      <td>business</td>
      <td>19.9900</td>
      <td>1991-06-12 00:00:00.000</td>
    </tr>
    <tr>
      <td>Straight Talk About Computers</td>
      <td>business</td>
      <td>19.9900</td>
      <td>1991-06-22 00:00:00.000</td>
    </tr>
    <tr>
      <td>Silicon Valley Gastronomic Treats</td>
      <td>mod_cook</td>
      <td>19.9900</td>
      <td>1991-06-09 00:00:00.000</td>
    </tr>
    <tr>
      <td>Prolonged Data Deprivation: Four Case Studies</td>
      <td>psychology</td>
      <td>19.9900</td>
      <td>1991-06-12 00:00:00.000</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">8 row(s)</p></div>

**Example 2.4 — where the cut falls matters.** The instructor repeats the query with `TOP 6 WITH TIES`. The point of the exercise: the number of rows returned is not simply *n*. It depends on whether row *n* ties with the rows after it.

In this data the ranking by price is 22.95, 21.59, 20.95, 20.00, then four books at 19.99, then lower prices. `TOP 4` ends exactly at a boundary (20.00 is followed by a different price), so it returns 4 rows. `TOP 5` and `TOP 6` both land *inside* the group of four 19.99 books, so each returns the whole group: 8 rows. The query below counts the rows each variant returns.

In [ ]:
SELECT 'TOP 4 WITH TIES' AS Query, COUNT(*) AS RowsReturned
FROM (SELECT TOP 4 WITH TIES t.price FROM dbo.titles AS t ORDER BY t.price DESC) AS x
UNION ALL
SELECT 'TOP 5 WITH TIES', COUNT(*)
FROM (SELECT TOP 5 WITH TIES t.price FROM dbo.titles AS t ORDER BY t.price DESC) AS x
UNION ALL
SELECT 'TOP 6 WITH TIES', COUNT(*)
FROM (SELECT TOP 6 WITH TIES t.price FROM dbo.titles AS t ORDER BY t.price DESC) AS x;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>Query</th>
      <th>RowsReturned</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>TOP 4 WITH TIES</td>
      <td>4</td>
    </tr>
    <tr>
      <td>TOP 5 WITH TIES</td>
      <td>8</td>
    </tr>
    <tr>
      <td>TOP 6 WITH TIES</td>
      <td>8</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">3 row(s)</p></div>

**Example 2.5 — `TOP ... PERCENT` and the rounding rule.** The first query computes how many rows to expect (30 percent of the table, rounded up). The second runs `TOP 30 PERCENT`. The counts match.

In [ ]:
SELECT COUNT(*) AS TotalBooks,
       CEILING(COUNT(*) * 0.30) AS Expected_Top30Percent
FROM dbo.titles AS t;

SELECT TOP 30 PERCENT t.title, t.price
FROM dbo.titles AS t
ORDER BY t.price DESC;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>TotalBooks</th>
      <th>Expected_Top30Percent</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>18</td>
      <td>6</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">1 row(s)</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title</th>
      <th>price</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>But Is It User Friendly?</td>
      <td>22.9500</td>
    </tr>
    <tr>
      <td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td>
      <td>21.5900</td>
    </tr>
    <tr>
      <td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td>
      <td>20.9500</td>
    </tr>
    <tr>
      <td>Secrets of Silicon Valley</td>
      <td>20.0000</td>
    </tr>
    <tr>
      <td>The Busy Executive's Database Guide</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>Straight Talk About Computers</td>
      <td>19.9900</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">6 row(s) — the sixth row is one of four books at 19.9900; which one is arbitrary</p></div>

**Example 2.6 — `DISTINCT` together with `TOP`.** `DISTINCT` is processed first, then `TOP` takes the first rows of the de-duplicated list. Without an `ORDER BY`, *which* three book types you get is not guaranteed. Adding the sort makes it meaningful.

In [ ]:
SELECT DISTINCT TOP 3 t.[type]
FROM dbo.titles AS t;

SELECT DISTINCT TOP 3 t.[type]
FROM dbo.titles AS t
ORDER BY t.[type];

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>type</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>business</td>
    </tr>
    <tr>
      <td>mod_cook</td>
    </tr>
    <tr>
      <td>UNDECIDED</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">3 row(s) — which three types you get is not guaranteed</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>type</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>business</td>
    </tr>
    <tr>
      <td>mod_cook</td>
    </tr>
    <tr>
      <td>popular_comp</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">3 row(s)</p></div>

**Example 2.7 — two related tools.**

*TOP with a variable.* `TOP (@n)` lets a stored procedure or script decide the row count at run time.

*OFFSET ... FETCH.* The ANSI-standard way to take a slice of an ordered result, and the basis of paging ("page 2 = rows 6 to 10"). Unlike `TOP`, it can skip rows. It requires `ORDER BY`; adding a unique column (`title_id`) as the last sort key keeps pages stable.

In [ ]:
DECLARE @n INT = 3;
SELECT TOP (@n) t.title, t.price
FROM dbo.titles AS t
ORDER BY t.price DESC;

-- Page 2 of the price ranking, five books per page
SELECT t.title, t.price
FROM dbo.titles AS t
ORDER BY t.price DESC, t.title_id
OFFSET 5 ROWS FETCH NEXT 5 ROWS ONLY;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title</th>
      <th>price</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>But Is It User Friendly?</td>
      <td>22.9500</td>
    </tr>
    <tr>
      <td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td>
      <td>21.5900</td>
    </tr>
    <tr>
      <td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td>
      <td>20.9500</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">3 row(s)</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title</th>
      <th>price</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Straight Talk About Computers</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>Silicon Valley Gastronomic Treats</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>Prolonged Data Deprivation: Four Case Studies</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>Sushi, Anyone?</td>
      <td>14.9900</td>
    </tr>
    <tr>
      <td>Cooking with Computers: Surreptitious Balance Sheets</td>
      <td>11.9500</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">5 row(s)</p></div>

**Example 2.8 — a preview of ranking (window functions).** The instructor's note says *Ranking ⇒ Window Function*, a topic for a later session. Here is a taste, because it explains `WITH TIES` precisely:

- `ROW_NUMBER()` gives every row a unique number, ties broken arbitrarily.
- `RANK()` gives tied rows the same number and then skips (1, 2, 2, 4).
- `DENSE_RANK()` gives tied rows the same number without skipping (1, 2, 2, 3).

`TOP 5 WITH TIES ... ORDER BY price DESC` returns exactly the rows whose `RANK()` is 5 or lower.

In [ ]:
SELECT t.title,
       t.price,
       ROW_NUMBER() OVER (ORDER BY t.price DESC) AS RowNum,
       RANK()       OVER (ORDER BY t.price DESC) AS Rnk,
       DENSE_RANK() OVER (ORDER BY t.price DESC) AS DenseRnk
FROM dbo.titles AS t
WHERE t.price IS NOT NULL
ORDER BY t.price DESC;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title</th>
      <th>price</th>
      <th>RowNum</th>
      <th>Rnk</th>
      <th>DenseRnk</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>But Is It User Friendly?</td>
      <td>22.9500</td>
      <td>1</td>
      <td>1</td>
      <td>1</td>
    </tr>
    <tr>
      <td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td>
      <td>21.5900</td>
      <td>2</td>
      <td>2</td>
      <td>2</td>
    </tr>
    <tr>
      <td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td>
      <td>20.9500</td>
      <td>3</td>
      <td>3</td>
      <td>3</td>
    </tr>
    <tr>
      <td>Secrets of Silicon Valley</td>
      <td>20.0000</td>
      <td>4</td>
      <td>4</td>
      <td>4</td>
    </tr>
    <tr>
      <td>The Busy Executive's Database Guide</td>
      <td>19.9900</td>
      <td>5</td>
      <td>5</td>
      <td>5</td>
    </tr>
    <tr>
      <td>Straight Talk About Computers</td>
      <td>19.9900</td>
      <td>6</td>
      <td>5</td>
      <td>5</td>
    </tr>
    <tr>
      <td>Silicon Valley Gastronomic Treats</td>
      <td>19.9900</td>
      <td>7</td>
      <td>5</td>
      <td>5</td>
    </tr>
    <tr>
      <td>Prolonged Data Deprivation: Four Case Studies</td>
      <td>19.9900</td>
      <td>8</td>
      <td>5</td>
      <td>5</td>
    </tr>
    <tr>
      <td>Sushi, Anyone?</td>
      <td>14.9900</td>
      <td>9</td>
      <td>9</td>
      <td>6</td>
    </tr>
    <tr>
      <td>Cooking with Computers: Surreptitious Balance Sheets</td>
      <td>11.9500</td>
      <td>10</td>
      <td>10</td>
      <td>7</td>
    </tr>
    <tr>
      <td>Fifty Years in Buckingham Palace Kitchens</td>
      <td>11.9500</td>
      <td>11</td>
      <td>10</td>
      <td>7</td>
    </tr>
    <tr>
      <td>Is Anger the Enemy?</td>
      <td>10.9500</td>
      <td>12</td>
      <td>12</td>
      <td>8</td>
    </tr>
    <tr>
      <td>Emotional Security: A New Algorithm</td>
      <td>7.9900</td>
      <td>13</td>
      <td>13</td>
      <td>9</td>
    </tr>
    <tr>
      <td>Life Without Fear</td>
      <td>7.0000</td>
      <td>14</td>
      <td>14</td>
      <td>10</td>
    </tr>
    <tr>
      <td>You Can Combat Computer Stress!</td>
      <td>2.9900</td>
      <td>15</td>
      <td>15</td>
      <td>11</td>
    </tr>
    <tr>
      <td>The Gourmet Microwave</td>
      <td>2.9900</td>
      <td>16</td>
      <td>15</td>
      <td>11</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">16 row(s) — RowNum among the four 19.9900 books is arbitrary; Rnk and DenseRnk are not</p></div>

### Pitfalls — `TOP`
- `TOP n` without `ORDER BY` is arbitrary. Always pair them unless you truly want "any n rows".
- `TOP n` can hide ties: two books at 19.99, only one is returned. Use `WITH TIES` when fairness matters.
- `PERCENT` rounds up, and the count depends on the *whole* table size.

---
## 3. Random sampling and unique identifiers

This section follows the instructor's example: *give me 70 percent of the customers, chosen at random*. Getting there requires a few ingredients: tidy customer columns, `TOP ... PERCENT`, and a way to shuffle rows (`NEWID()`), which leads naturally to the topic of GUIDs and how they differ from `IDENTITY` columns.

### 3.1 The customer query
`dbo.DimCustomer` in `AdventureWorksDW2022` has 18,484 customers. The instructor builds a tidy customer list with a full name and an age. Two techniques appear:

- **`CONCAT_WS(separator, a, b, c)`** joins values with a separator and **skips `NULL`s**. The older form `FirstName + ' ' + MiddleName + ' ' + LastName` returns `NULL` for the *whole* name whenever the middle name is `NULL`, which is why the instructor comments it out. (`CONCAT_WS` needs SQL Server 2017 or later.)
- **`DATEDIFF(year, BirthDate, GETDATE())`** counts how many calendar-year boundaries lie between two dates. That is only an *approximation* of age: someone born in October is counted a year older from 1 January.

**Example 3.1 — string concatenation with `+` versus `CONCAT_WS`.** We filter to customers without a middle name to make the difference visible.

In [ ]:
USE AdventureWorksDW2022;

SELECT TOP 5
       dc.CustomerKey,
       dc.FirstName + ' ' + dc.MiddleName + ' ' + dc.LastName   AS FullName_Plus,
       CONCAT_WS(' ', dc.FirstName, dc.MiddleName, dc.LastName) AS FullName_ConcatWS
FROM dbo.DimCustomer AS dc
WHERE dc.MiddleName IS NULL
ORDER BY dc.CustomerKey;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>CustomerKey</th>
      <th>FullName_Plus</th>
      <th>FullName_ConcatWS</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>11002</td>
      <td>NULL</td>
      <td>Ruben Torres</td>
    </tr>
    <tr>
      <td>11003</td>
      <td>NULL</td>
      <td>Christy Zhu</td>
    </tr>
    <tr>
      <td>11004</td>
      <td>NULL</td>
      <td>Elizabeth Johnson</td>
    </tr>
    <tr>
      <td>11005</td>
      <td>NULL</td>
      <td>Julio Ruiz</td>
    </tr>
    <tr>
      <td>11007</td>
      <td>NULL</td>
      <td>Marco Mehta</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">5 row(s)</p></div>

**Example 3.2 — the instructor's customer list.** The original returns all 18,484 rows; we keep the first 10 so the output stays readable. The second query shows an exact age: it subtracts 1 when this year's birthday has not happened yet.

In [ ]:
SELECT TOP 10
       dc.CustomerKey,
       CONCAT_WS(' ', dc.FirstName, dc.MiddleName, dc.LastName) AS FullName,
       dc.EnglishEducation,
       dc.EnglishOccupation,
       dc.BirthDate,
       DATEDIFF(year, dc.BirthDate, GETDATE()) AS Age,
       dc.CommuteDistance
FROM dbo.DimCustomer AS dc
ORDER BY dc.CustomerKey;

-- Age corrected for birthdays that have not happened yet this year
SELECT TOP 5
       dc.CustomerKey,
       dc.BirthDate,
       DATEDIFF(year, dc.BirthDate, GETDATE()) AS Age_Approx,
       DATEDIFF(year, dc.BirthDate, GETDATE())
         - CASE WHEN DATEADD(year, DATEDIFF(year, dc.BirthDate, GETDATE()), dc.BirthDate) > GETDATE()
                THEN 1 ELSE 0 END                AS Age_Exact
FROM dbo.DimCustomer AS dc
ORDER BY dc.CustomerKey;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>CustomerKey</th>
      <th>FullName</th>
      <th>EnglishEducation</th>
      <th>EnglishOccupation</th>
      <th>BirthDate</th>
      <th>Age</th>
      <th>CommuteDistance</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>11000</td>
      <td>Jon V Yang</td>
      <td>Bachelors</td>
      <td>Professional</td>
      <td>1971-10-06</td>
      <td>55</td>
      <td>1-2 Miles</td>
    </tr>
    <tr>
      <td>11001</td>
      <td>Eugene L Huang</td>
      <td>Bachelors</td>
      <td>Professional</td>
      <td>1976-05-10</td>
      <td>50</td>
      <td>0-1 Miles</td>
    </tr>
    <tr>
      <td>11002</td>
      <td>Ruben Torres</td>
      <td>Bachelors</td>
      <td>Professional</td>
      <td>1971-02-09</td>
      <td>55</td>
      <td>2-5 Miles</td>
    </tr>
    <tr>
      <td>11003</td>
      <td>Christy Zhu</td>
      <td>Bachelors</td>
      <td>Professional</td>
      <td>1973-08-14</td>
      <td>53</td>
      <td>5-10 Miles</td>
    </tr>
    <tr>
      <td>11004</td>
      <td>Elizabeth Johnson</td>
      <td>Bachelors</td>
      <td>Professional</td>
      <td>1979-08-05</td>
      <td>47</td>
      <td>1-2 Miles</td>
    </tr>
    <tr>
      <td>11005</td>
      <td>Julio Ruiz</td>
      <td>Bachelors</td>
      <td>Professional</td>
      <td>1976-08-01</td>
      <td>50</td>
      <td>5-10 Miles</td>
    </tr>
    <tr>
      <td>11006</td>
      <td>Janet G Alvarez</td>
      <td>Bachelors</td>
      <td>Professional</td>
      <td>1976-12-02</td>
      <td>50</td>
      <td>5-10 Miles</td>
    </tr>
    <tr>
      <td>11007</td>
      <td>Marco Mehta</td>
      <td>Bachelors</td>
      <td>Professional</td>
      <td>1969-11-06</td>
      <td>57</td>
      <td>0-1 Miles</td>
    </tr>
    <tr>
      <td>11008</td>
      <td>Rob Verhoff</td>
      <td>Bachelors</td>
      <td>Professional</td>
      <td>1975-07-04</td>
      <td>51</td>
      <td>10+ Miles</td>
    </tr>
    <tr>
      <td>11009</td>
      <td>Shannon C Carlson</td>
      <td>Bachelors</td>
      <td>Professional</td>
      <td>1969-09-29</td>
      <td>57</td>
      <td>5-10 Miles</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">10 row(s)</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>CustomerKey</th>
      <th>BirthDate</th>
      <th>Age_Approx</th>
      <th>Age_Exact</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>11000</td>
      <td>1971-10-06</td>
      <td>55</td>
      <td>54</td>
    </tr>
    <tr>
      <td>11001</td>
      <td>1976-05-10</td>
      <td>50</td>
      <td>50</td>
    </tr>
    <tr>
      <td>11002</td>
      <td>1971-02-09</td>
      <td>55</td>
      <td>55</td>
    </tr>
    <tr>
      <td>11003</td>
      <td>1973-08-14</td>
      <td>53</td>
      <td>53</td>
    </tr>
    <tr>
      <td>11004</td>
      <td>1979-08-05</td>
      <td>47</td>
      <td>47</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">5 row(s) — Age_Approx counts calendar-year boundaries, so it overstates the age until the birthday has passed</p></div>

### 3.2 `TOP 70 PERCENT` is *not* a random sample
`TOP 70 PERCENT` with no `ORDER BY` returns the first 70 percent of rows in whatever order SQL Server reads them. It is *a* subset, not a *random* one, and it tends to be the same subset every time. The count is predictable: 70 percent of 18,484 is 12,938.8, rounded up to **12,939**.

In [ ]:
SELECT COUNT(*) AS TotalCustomers,
       CEILING(COUNT(*) * 0.70) AS Expected_Top70Percent
FROM dbo.DimCustomer;

SELECT TOP 70 PERCENT
       dc.CustomerKey,
       CONCAT_WS(' ', dc.FirstName, dc.MiddleName, dc.LastName) AS FullName,
       dc.EnglishEducation, dc.EnglishOccupation, dc.BirthDate,
       DATEDIFF(year, dc.BirthDate, GETDATE()) AS Age,
       dc.CommuteDistance
FROM dbo.DimCustomer AS dc;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>TotalCustomers</th>
      <th>Expected_Top70Percent</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>18484</td>
      <td>12939</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">1 row(s)</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>CustomerKey</th>
      <th>FullName</th>
      <th>EnglishEducation</th>
      <th>EnglishOccupation</th>
      <th>BirthDate</th>
      <th>Age</th>
      <th>CommuteDistance</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>11000</td>
      <td>Jon V Yang</td>
      <td>Bachelors</td>
      <td>Professional</td>
      <td>1971-10-06</td>
      <td>55</td>
      <td>1-2 Miles</td>
    </tr>
    <tr>
      <td>11001</td>
      <td>Eugene L Huang</td>
      <td>Bachelors</td>
      <td>Professional</td>
      <td>1976-05-10</td>
      <td>50</td>
      <td>0-1 Miles</td>
    </tr>
    <tr>
      <td>11002</td>
      <td>Ruben Torres</td>
      <td>Bachelors</td>
      <td>Professional</td>
      <td>1971-02-09</td>
      <td>55</td>
      <td>2-5 Miles</td>
    </tr>
    <tr>
      <td>11003</td>
      <td>Christy Zhu</td>
      <td>Bachelors</td>
      <td>Professional</td>
      <td>1973-08-14</td>
      <td>53</td>
      <td>5-10 Miles</td>
    </tr>
    <tr>
      <td>11004</td>
      <td>Elizabeth Johnson</td>
      <td>Bachelors</td>
      <td>Professional</td>
      <td>1979-08-05</td>
      <td>47</td>
      <td>1-2 Miles</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">12939 row(s) — first 5 shown — the first 70% in whatever order SQL Server read them; not random</p></div>

### 3.3 Identity columns versus GUIDs
To shuffle rows we need a different random value on each row. SQL Server provides one: `NEWID()`, which generates a **GUID**. First, the background.

| | `IDENTITY` | GUID (`UNIQUEIDENTIFIER`) |
|---|---|---|
| What it is | An automatic counter: `IDENTITY(seed, increment)`, usually `IDENTITY(1,1)` | A 16-byte globally unique value |
| Example | `1, 2, 3, ...` | `B2DD48D7-F2A0-4DF4-BA2B-B420B16365B4` |
| Unique across | one table | any table, database, server and time |
| Size | 4 bytes (`INT`) | **16 bytes** |
| Generated by | the table itself, on insert | `NEWID()` (random) or `NEWSEQUENTIALID()` (increasing) |
| Typical use | surrogate primary key of a table | merging data from several systems, distributed apps, ids you do not want to be guessable, random shuffling |

**GUID.** *Globally Unique IDentifier*; the name explains itself: GU = Globally Unique, ID = Identifier. It is stored in 16 bytes but *displayed* as 36 characters: 32 hexadecimal digits plus 4 hyphens. The instructor's note about the *NIC* (network interface card) and *MAC address* refers to history: older GUID algorithms mixed in the physical address of the machine's network card to guarantee uniqueness. `NEWID()` in SQL Server produces random values today.

**What makes a good primary key?** The instructor lists the properties: *Unique*, *Not NULL*, a *narrow* data type, and *not updated*. `IDENTITY` integers satisfy all four. A GUID is unique and not null but wide (16 bytes versus 4), and a random GUID inserts rows at random positions in the clustered index, causing page splits and fragmentation. That is exactly the problem `NEWSEQUENTIALID()` was designed to reduce.

**Example 3.3 — `NEWID()`: a new value on every call, on every row.** Run it twice and the values never repeat (the values below are one example run). In the second query, `NEWID()` is evaluated *once per row*, so every row gets its own GUID. This per-row behaviour is what makes `ORDER BY NEWID()` shuffle the table.

In [ ]:
SELECT NEWID() AS [GUID];

SELECT TOP 5 dc.CustomerKey, NEWID() AS [GUID]
FROM dbo.DimCustomer AS dc;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>GUID</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>F38B2FFC-80A4-4F5A-91C9-BC701E7EA419</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">1 row(s) — random: differs on every run</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>CustomerKey</th>
      <th>GUID</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>11000</td>
      <td>F3F49249-DC28-4F90-A5AE-C7978306D03B</td>
    </tr>
    <tr>
      <td>11001</td>
      <td>E5121482-3929-4D22-A255-ACCB1A466884</td>
    </tr>
    <tr>
      <td>11002</td>
      <td>6BAD6BE2-8E7A-46E9-9F19-950499DD251D</td>
    </tr>
    <tr>
      <td>11003</td>
      <td>D7A7A3CC-8C3D-4F16-9293-DE8FC88B2875</td>
    </tr>
    <tr>
      <td>11004</td>
      <td>7DABE929-C4A3-44BF-86CD-75E9BB049A79</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">5 row(s) — random: one GUID per row, different every run</p></div>

**Example 3.4 — 16 bytes, but 36 characters.** `DATALENGTH` reports the storage size; `LEN` of the text form reports the displayed length. The instructor's `SELECT LEN('B2DD48D7-...')` demonstrates the 36.

In [ ]:
SELECT DATALENGTH(NEWID())                  AS StorageBytes,
       LEN(CAST(NEWID() AS VARCHAR(36)))    AS DisplayedCharacters,
       LEN('B2DD48D7-F2A0-4DF4-BA2B-B420B16365B4') AS InstructorsExample;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>StorageBytes</th>
      <th>DisplayedCharacters</th>
      <th>InstructorsExample</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>16</td>
      <td>36</td>
      <td>36</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">1 row(s)</p></div>

### 3.4 The random sample: `ORDER BY NEWID()`
Attach a fresh GUID to every row, sort by it, and the order is effectively random. Combine with `TOP` to take a sample. Run the first cell several times in SSMS to see different customers each time; the output below is one example run (uniqueidentifier values sort by an internal byte order, so the GUIDs will not look alphabetical).

**How to read the instructor's output.** The two sample lines in his notes (customer 11000, *Jon V Yang*, with different GUIDs) show the same customer receiving a *different* GUID on each execution. That is the proof that the GUID is regenerated on every run and that the sample changes with it.

In [ ]:
-- Five random customers
SELECT TOP 5
       dc.CustomerKey,
       CONCAT_WS(' ', dc.FirstName, dc.MiddleName, dc.LastName) AS FullName,
       NEWID() AS [GUID]
FROM dbo.DimCustomer AS dc
ORDER BY [GUID];

-- The goal of the instructor's example: a random 70 percent sample
SELECT TOP 70 PERCENT
       dc.CustomerKey,
       CONCAT_WS(' ', dc.FirstName, dc.MiddleName, dc.LastName) AS FullName,
       dc.EnglishEducation, dc.EnglishOccupation, dc.BirthDate,
       DATEDIFF(year, dc.BirthDate, GETDATE()) AS Age,
       dc.CommuteDistance
FROM dbo.DimCustomer AS dc
ORDER BY NEWID();

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>CustomerKey</th>
      <th>FullName</th>
      <th>GUID</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>12356</td>
      <td>Julia L Wood</td>
      <td>70EB9A0A-9626-4AE6-85E8-18FAC0433CBD</td>
    </tr>
    <tr>
      <td>27379</td>
      <td>Isabel Ross</td>
      <td>14AA4E71-9D3C-4DEC-80A6-1F933D6C51E3</td>
    </tr>
    <tr>
      <td>15929</td>
      <td>Karla V Raji</td>
      <td>5071950E-ADEC-4F11-BD83-6E77AF67D461</td>
    </tr>
    <tr>
      <td>24922</td>
      <td>Naomi Jimenez</td>
      <td>1919E93A-D117-45AD-8988-93101C593AF5</td>
    </tr>
    <tr>
      <td>14384</td>
      <td>Rachel F Miller</td>
      <td>E4163207-D094-4996-82F0-EE99731C9452</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">5 row(s) — random: different customers on every run</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>CustomerKey</th>
      <th>FullName</th>
      <th>EnglishEducation</th>
      <th>EnglishOccupation</th>
      <th>BirthDate</th>
      <th>Age</th>
      <th>CommuteDistance</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>12356</td>
      <td>Julia L Wood</td>
      <td>Bachelors</td>
      <td>Professional</td>
      <td>1977-02-18</td>
      <td>49</td>
      <td>10+ Miles</td>
    </tr>
    <tr>
      <td>27379</td>
      <td>Isabel Ross</td>
      <td>High School</td>
      <td>Professional</td>
      <td>1952-12-06</td>
      <td>74</td>
      <td>1-2 Miles</td>
    </tr>
    <tr>
      <td>24922</td>
      <td>Naomi Jimenez</td>
      <td>Partial High School</td>
      <td>Professional</td>
      <td>1975-10-06</td>
      <td>51</td>
      <td>10+ Miles</td>
    </tr>
    <tr>
      <td>14384</td>
      <td>Rachel F Miller</td>
      <td>Bachelors</td>
      <td>Management</td>
      <td>1950-11-19</td>
      <td>76</td>
      <td>2-5 Miles</td>
    </tr>
    <tr>
      <td>15929</td>
      <td>Karla V Raji</td>
      <td>High School</td>
      <td>Professional</td>
      <td>1962-02-07</td>
      <td>64</td>
      <td>5-10 Miles</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">12939 row(s) — first 5 shown — random: same 12,939 rows as before in count, but a random set</p></div>

**Note on cost and an alternative.** `ORDER BY NEWID()` must generate a GUID for every row and sort the whole table, which is fine for 18,484 rows but slow for hundreds of millions. `TABLESAMPLE` is much faster because it picks random *pages*, but it is approximate: the number of rows returned varies from run to run. Run it twice in SSMS and compare; the two counts below are illustrative.

In [ ]:
SELECT COUNT(*) AS SampledRows FROM dbo.DimCustomer TABLESAMPLE (10 PERCENT);
SELECT COUNT(*) AS SampledRows FROM dbo.DimCustomer TABLESAMPLE (10 PERCENT);

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>SampledRows</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>1850</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">1 row(s) — random and approximate; illustrative value, yours will differ</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>SampledRows</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>1925</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">1 row(s) — random and approximate; illustrative value, yours will differ</p></div>

### 3.5 `NEWSEQUENTIALID()` and `IDENTITY` in a real table
`NEWSEQUENTIALID()` creates GUIDs that increase over time, which avoids most index fragmentation. The instructor's note shows it called with a bare `SELECT NEWSEQUENTIALID()`. **That statement fails**: `NEWSEQUENTIALID()` can only be used as a `DEFAULT` for a `UNIQUEIDENTIFIER` column. The first cell shows the error; the second shows the correct usage side by side with an `IDENTITY` column.

In [ ]:
SELECT NEWSEQUENTIALID();

Msg 302, Level 16, State 0, Line 1
The newsequentialid() built-in function can only be used in a DEFAULT expression for a column of type 'uniqueidentifier' in a CREATE/ALTER TABLE statement. It cannot be combined with other operators to form a complex scalar expression.

In [ ]:
-- One batch = one session, so the temporary table (#Demo) lives long enough to use.
CREATE TABLE #Demo
(
    RowId   INT IDENTITY(1, 1)                          NOT NULL,   -- seed 1, increment 1
    RowGuid UNIQUEIDENTIFIER DEFAULT NEWSEQUENTIALID()  NOT NULL,
    Note    VARCHAR(20)
);

INSERT INTO #Demo (Note) VALUES ('first'), ('second'), ('third');

SELECT * FROM #Demo ORDER BY RowGuid;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>RowId</th>
      <th>RowGuid</th>
      <th>Note</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>1</td>
      <td>35D14880-3B7A-EF11-9C4D-0A1B2C3D4E5F</td>
      <td>first</td>
    </tr>
    <tr>
      <td>2</td>
      <td>36D14880-3B7A-EF11-9C4D-0A1B2C3D4E5F</td>
      <td>second</td>
    </tr>
    <tr>
      <td>3</td>
      <td>37D14880-3B7A-EF11-9C4D-0A1B2C3D4E5F</td>
      <td>third</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">3 row(s) — RowId is always 1, 2, 3; the GUID values are random-looking but increase, and differ on every run</p></div>

### Pitfalls — random sampling and GUIDs
- `TOP 70 PERCENT` alone is not random. You need `ORDER BY NEWID()`.
- `NEWID()` is different on every execution, so a "random" result cannot be reproduced. Save it (see `SELECT ... INTO` below) if you need the same sample again.
- `NEWSEQUENTIALID()` works only in a column `DEFAULT`.
- GUIDs are four times wider than an `INT`; every index and every foreign key pointing to them inherits that size.

---
## 4. Fully qualified names

### Purpose
Every object in SQL Server has a four-part address:

`server . database . schema . object`

Normally you work inside one database (`USE pubs`) and the shorter name `dbo.titles` is enough. A **fully qualified name** lets you reach a table in *another* database without switching to it, which is essential for cross-database queries and for scripts that must not depend on the current database.

### Syntax
```sql
SELECT ... FROM database.schema.table;      -- fully qualified:  pubs.dbo.titles
SELECT ... FROM database..table;            -- schema omitted:   pubs..titles
```
The double dot `..` means "use the default schema" (normally `dbo`). It works, but the explicit `database.schema.table` form is clearer and is considered better practice. A fourth part in front (`[server].database.schema.table`) reaches a *linked server*. Square brackets, as in `[type]` or `[GUID]`, protect names that are keywords or contain spaces.

In [ ]:
-- Connected to master: the double-dot form from the instructor's note
USE master;
SELECT TOP 5 t.title, t.[type], t.pubdate
FROM pubs..titles AS t
ORDER BY t.price DESC;

-- Same thing with the explicit schema (preferred)
SELECT TOP 5 t.title, t.[type], t.pubdate
FROM pubs.dbo.titles AS t
ORDER BY t.price DESC;

-- Connected to pubs, reading from another database
USE pubs;
SELECT TOP 3 dc.CustomerKey, dc.FirstName, dc.LastName
FROM AdventureWorksDW2022.dbo.DimCustomer AS dc
ORDER BY dc.CustomerKey;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title</th>
      <th>type</th>
      <th>pubdate</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>But Is It User Friendly?</td>
      <td>popular_comp</td>
      <td>1991-06-30 00:00:00.000</td>
    </tr>
    <tr>
      <td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td>
      <td>psychology</td>
      <td>1991-10-21 00:00:00.000</td>
    </tr>
    <tr>
      <td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td>
      <td>trad_cook</td>
      <td>1991-10-21 00:00:00.000</td>
    </tr>
    <tr>
      <td>Secrets of Silicon Valley</td>
      <td>popular_comp</td>
      <td>1994-06-12 00:00:00.000</td>
    </tr>
    <tr>
      <td>The Busy Executive's Database Guide</td>
      <td>business</td>
      <td>1991-06-12 00:00:00.000</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">5 row(s)</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title</th>
      <th>type</th>
      <th>pubdate</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>But Is It User Friendly?</td>
      <td>popular_comp</td>
      <td>1991-06-30 00:00:00.000</td>
    </tr>
    <tr>
      <td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td>
      <td>psychology</td>
      <td>1991-10-21 00:00:00.000</td>
    </tr>
    <tr>
      <td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td>
      <td>trad_cook</td>
      <td>1991-10-21 00:00:00.000</td>
    </tr>
    <tr>
      <td>Secrets of Silicon Valley</td>
      <td>popular_comp</td>
      <td>1994-06-12 00:00:00.000</td>
    </tr>
    <tr>
      <td>The Busy Executive's Database Guide</td>
      <td>business</td>
      <td>1991-06-12 00:00:00.000</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">5 row(s)</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>CustomerKey</th>
      <th>FirstName</th>
      <th>LastName</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>11000</td>
      <td>Jon</td>
      <td>Yang</td>
    </tr>
    <tr>
      <td>11001</td>
      <td>Eugene</td>
      <td>Huang</td>
    </tr>
    <tr>
      <td>11002</td>
      <td>Ruben</td>
      <td>Torres</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">3 row(s)</p></div>

---
## 5. Making-table queries: `SELECT ... INTO`

### Purpose
The instructor's one-line summary is *Query ⇒ Select ⇒ Table*: a `SELECT` normally displays rows and forgets them; adding `INTO` makes the result **persist** as a brand-new table. You create the table and fill it in one step, without writing `CREATE TABLE` and `INSERT`. Common uses: a quick backup of a table before changing it, a snapshot of a random sample, a staging table, a copy of data into another database, and temporary tables (`#name`).

### Syntax
```sql
SELECT column_list
INTO   new_table
FROM   ...
[WHERE ... ] [ORDER BY ...];
```

### What is copied, and what is not
- **Copied:** column names (or their aliases), data types, nullability and the data.
- **Not copied:** primary keys, foreign keys, other constraints, defaults and indexes. You add them afterwards with `ALTER TABLE`. The instructor's remark *Constraint: Primary Key, Foreign Key* refers to exactly this.
- `INTO` fails if the target table already exists.
- `INTO #name` creates a temporary table that disappears at the end of the session; `INTO ##name` is a global temporary table.
- The target can be in another database: `INTO pubs..MyTable`.
- `ORDER BY` is allowed, but a table is still a set. Do not assume the stored rows keep that order; always sort again when you read.

**Example 5.1 — the instructor's `Sematec_Books`.** The columns are renamed with aliases, `pubdate` is converted from date-time to plain `DATE`, and the result is stored as a new table. (We use the name `Sematec_Books_NB` so we do not overwrite the table you created in class. `DROP TABLE IF EXISTS` makes the cell safe to re-run.)

In [ ]:
USE pubs;
DROP TABLE IF EXISTS dbo.Sematec_Books_NB;

SELECT t.title_id               AS Book_Code,
       t.title                  AS Book_Name,
       t.price                  AS Book_Price,
       CAST(t.pubdate AS DATE)  AS Publish_Date
INTO   dbo.Sematec_Books_NB
FROM   dbo.titles AS t
ORDER BY Book_Price DESC;

-- The result now persists as an ordinary table
SELECT * FROM dbo.Sematec_Books_NB;

(18 rows affected)

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>Book_Code</th>
      <th>Book_Name</th>
      <th>Book_Price</th>
      <th>Publish_Date</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>PC1035</td>
      <td>But Is It User Friendly?</td>
      <td>22.9500</td>
      <td>1991-06-30</td>
    </tr>
    <tr>
      <td>PS1372</td>
      <td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td>
      <td>21.5900</td>
      <td>1991-10-21</td>
    </tr>
    <tr>
      <td>TC3218</td>
      <td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td>
      <td>20.9500</td>
      <td>1991-10-21</td>
    </tr>
    <tr>
      <td>PC8888</td>
      <td>Secrets of Silicon Valley</td>
      <td>20.0000</td>
      <td>1994-06-12</td>
    </tr>
    <tr>
      <td>BU1032</td>
      <td>The Busy Executive's Database Guide</td>
      <td>19.9900</td>
      <td>1991-06-12</td>
    </tr>
    <tr>
      <td>BU7832</td>
      <td>Straight Talk About Computers</td>
      <td>19.9900</td>
      <td>1991-06-22</td>
    </tr>
    <tr>
      <td>MC2222</td>
      <td>Silicon Valley Gastronomic Treats</td>
      <td>19.9900</td>
      <td>1991-06-09</td>
    </tr>
    <tr>
      <td>PS3333</td>
      <td>Prolonged Data Deprivation: Four Case Studies</td>
      <td>19.9900</td>
      <td>1991-06-12</td>
    </tr>
    <tr>
      <td>TC7777</td>
      <td>Sushi, Anyone?</td>
      <td>14.9900</td>
      <td>1991-06-12</td>
    </tr>
    <tr>
      <td>BU1111</td>
      <td>Cooking with Computers: Surreptitious Balance Sheets</td>
      <td>11.9500</td>
      <td>1991-06-09</td>
    </tr>
    <tr>
      <td>TC4203</td>
      <td>Fifty Years in Buckingham Palace Kitchens</td>
      <td>11.9500</td>
      <td>1991-06-12</td>
    </tr>
    <tr>
      <td>PS2091</td>
      <td>Is Anger the Enemy?</td>
      <td>10.9500</td>
      <td>1991-06-15</td>
    </tr>
    <tr>
      <td>PS7777</td>
      <td>Emotional Security: A New Algorithm</td>
      <td>7.9900</td>
      <td>1991-06-12</td>
    </tr>
    <tr>
      <td>PS2106</td>
      <td>Life Without Fear</td>
      <td>7.0000</td>
      <td>1991-10-05</td>
    </tr>
    <tr>
      <td>BU2075</td>
      <td>You Can Combat Computer Stress!</td>
      <td>2.9900</td>
      <td>1991-06-30</td>
    </tr>
    <tr>
      <td>MC3021</td>
      <td>The Gourmet Microwave</td>
      <td>2.9900</td>
      <td>1991-06-18</td>
    </tr>
    <tr>
      <td>MC3026</td>
      <td>The Psychology of Computer Cooking</td>
      <td>NULL</td>
      <td>2026-10-03</td>
    </tr>
    <tr>
      <td>PC9999</td>
      <td>Net Etiquette</td>
      <td>NULL</td>
      <td>2026-10-03</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">18 row(s) — SELECT * has no ORDER BY, so the stored order is not guaranteed</p></div>

**Example 5.2 — what did `INTO` copy?** The column list shows names and types. Then we check for constraints: there are none yet.

In [ ]:
SELECT c.COLUMN_NAME, c.DATA_TYPE, c.CHARACTER_MAXIMUM_LENGTH
FROM INFORMATION_SCHEMA.COLUMNS AS c
WHERE c.TABLE_NAME = 'Sematec_Books_NB'
ORDER BY c.ORDINAL_POSITION;

SELECT kc.name AS ConstraintName, kc.type_desc
FROM sys.key_constraints AS kc
WHERE kc.parent_object_id = OBJECT_ID('dbo.Sematec_Books_NB');

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>COLUMN_NAME</th>
      <th>DATA_TYPE</th>
      <th>CHARACTER_MAXIMUM_LENGTH</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Book_Code</td>
      <td>char</td>
      <td>6.0</td>
    </tr>
    <tr>
      <td>Book_Name</td>
      <td>varchar</td>
      <td>80.0</td>
    </tr>
    <tr>
      <td>Book_Price</td>
      <td>money</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>Publish_Date</td>
      <td>date</td>
      <td>NULL</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">4 row(s)</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>ConstraintName</th>
      <th>type_desc</th>
    </tr>
  </thead>
  <tbody>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">0 row(s) — no constraints were copied</p></div>

**Example 5.3 — adding the missing constraint.** The new table has no primary key. `Book_Code` is unique and not null, so it qualifies; we add the key with `ALTER TABLE`.

In [ ]:
ALTER TABLE dbo.Sematec_Books_NB
ADD CONSTRAINT PK_Sematec_Books_NB PRIMARY KEY (Book_Code);

SELECT kc.name AS ConstraintName, kc.type_desc
FROM sys.key_constraints AS kc
WHERE kc.parent_object_id = OBJECT_ID('dbo.Sematec_Books_NB');

Commands completed successfully.

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>ConstraintName</th>
      <th>type_desc</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>PK_Sematec_Books_NB</td>
      <td>PRIMARY_KEY_CONSTRAINT</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">1 row(s)</p></div>

**Example 5.4 — three more patterns.**

1. *Copy the structure only*: `WHERE 1 = 0` is never true, so no rows are copied but the columns are created.
2. *Copy data from another database*: the instructor's `SELECT * INTO pubs..MyTable FROM SampleTable` (run from a database called `DB1`) does this. Here we copy 100 customers from `AdventureWorksDW2022` into `pubs`.
3. *Temporary table*: `#ExpensiveBooks` exists only for this session, which makes it ideal for intermediate results.

In [ ]:
-- 1. Structure only
DROP TABLE IF EXISTS dbo.Sematec_Books_Empty_NB;
SELECT * INTO dbo.Sematec_Books_Empty_NB
FROM dbo.Sematec_Books_NB
WHERE 1 = 0;

SELECT COUNT(*) AS NumRows FROM dbo.Sematec_Books_Empty_NB;

-- 2. Copy across databases
DROP TABLE IF EXISTS dbo.MyCustomerCopy_NB;
SELECT TOP 100 c.CustomerKey, c.FirstName, c.LastName
INTO   pubs.dbo.MyCustomerCopy_NB
FROM   AdventureWorksDW2022.dbo.DimCustomer AS c
ORDER BY c.CustomerKey;

SELECT COUNT(*) AS NumRows FROM pubs.dbo.MyCustomerCopy_NB;

-- 3. Temporary table
SELECT t.title, t.price
INTO   #ExpensiveBooks
FROM   dbo.titles AS t
WHERE  t.price > 15;

SELECT * FROM #ExpensiveBooks ORDER BY price DESC;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>NumRows</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>0</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">1 row(s)</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>NumRows</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>100</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">1 row(s)</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>title</th>
      <th>price</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>But Is It User Friendly?</td>
      <td>22.9500</td>
    </tr>
    <tr>
      <td>Computer Phobic AND Non-Phobic Individuals: Behavior Variations</td>
      <td>21.5900</td>
    </tr>
    <tr>
      <td>Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean</td>
      <td>20.9500</td>
    </tr>
    <tr>
      <td>Secrets of Silicon Valley</td>
      <td>20.0000</td>
    </tr>
    <tr>
      <td>The Busy Executive's Database Guide</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>Straight Talk About Computers</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>Silicon Valley Gastronomic Treats</td>
      <td>19.9900</td>
    </tr>
    <tr>
      <td>Prolonged Data Deprivation: Four Case Studies</td>
      <td>19.9900</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">8 row(s) — order among the four 19.9900 books is arbitrary</p></div>

**Example 5.5 — freezing a random sample.** Because `NEWID()` gives a different result each time, `SELECT ... INTO` is the way to keep one particular random sample, for example a training set.

In [ ]:
DROP TABLE IF EXISTS dbo.CustomerSample_NB;

SELECT TOP 70 PERCENT dc.CustomerKey, dc.EnglishEducation, dc.EnglishOccupation
INTO   pubs.dbo.CustomerSample_NB
FROM   AdventureWorksDW2022.dbo.DimCustomer AS dc
ORDER BY NEWID();

SELECT COUNT(*) AS SampleSize FROM pubs.dbo.CustomerSample_NB;

(12939 rows affected)

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>SampleSize</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>12939</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">1 row(s)</p></div>

### Pitfalls — `SELECT INTO`
- The target table must not exist yet; use `DROP TABLE IF EXISTS` first, or use `INSERT INTO ... SELECT` to append to an existing table.
- No keys, constraints or indexes come along.
- Computed values with ambiguous types (such as a bare `NULL`) produce a surprising column type; use `CAST`.
- In the instructor's cross-database example, `DB1` and `SampleTable` must exist on your server for it to run.

---
## 6. Set operators: `UNION ALL`, `UNION`, `INTERSECT`, `EXCEPT`

### Purpose
The instructor starts from E. F. Codd's relational model: relational databases are built on **set theory**. A *join* combines tables **side by side** (adding columns). A *set operator* stacks the results of two queries **on top of each other** (adding rows). Use them when the same kind of data lives in several tables, such as the locations of authors, stores and publishers, and you want one combined list, the overlap, or the difference.

### The four operators, with the instructor's sets
Take A = {10, 50, 30} and B = {90, 20, 30, 50}.

| Operation | Maths | SQL | Result for A, B | Duplicates |
|---|---|---|---|---|
| Intersection | A ∩ B | `INTERSECT` | {30, 50} | removed |
| Difference | A − B = A − (A ∩ B) | `EXCEPT` | {10} | removed |
| Union with duplicates | A ∪ B, as a bag | `UNION ALL` | {10, 20, 30, 30, 50, 50, 90} | **kept** |
| Union | A ∪ B | `UNION` | {10, 20, 30, 50, 90} | removed |

### Rules for combining queries
1. Both queries must return the **same number of columns**.
2. Columns are matched **by position, not by name**, and their data types must be compatible.
3. The **column names of the result come from the first query**.
4. `ORDER BY` may appear **once, at the very end**, and refers to the first query's column names.
5. `UNION`, `INTERSECT` and `EXCEPT` remove duplicate rows, and treat two `NULL`s as equal. `UNION ALL` does no de-duplication, so it is faster; prefer it when duplicates are impossible or wanted.
6. **Precedence:** `INTERSECT` binds tighter than `UNION` and `EXCEPT`; the others are evaluated left to right. Use parentheses to be explicit.

# ترکیب عمودی جداول (Vertical Combination) در SQL Server

ترکیب عمودی یعنی **زیر هم گذاشتن سطرهای چند Query** (برخلاف JOIN که ستون‌ها را کنار هم می‌گذارد). عملگرهای آن چهار تا هستند: `UNION ALL`، `UNION`، `INTERSECT` و `EXCEPT`. این‌ها مستقیماً از **نظریه مجموعه‌ها** در مدل رابطه‌ای کاد (F. Codd) آمده‌اند، همان چیزی که بالای کد خودتان با مجموعه‌های A و B نوشته‌اید.

## قوانین مشترک

1. **تعداد ستون‌ها** در همه Queryها باید برابر باشد.
2. **نوع داده ستون‌های متناظر** باید سازگار باشد (ستون اول با ستون اول، دوم با دوم و...).
3. **نام ستون‌های خروجی** از Query اول گرفته می‌شود (در کد شما `Shalgham` به عنوان alias در Query دوم نادیده گرفته می‌شود).
4. اگر یک Query ستونی ندارد، با `NULL` پر می‌شود، مثل `NULL as zip` برای publishers.
5. `ORDER BY` فقط یک بار و در **انتهای** کل ترکیب می‌آید.

## ۱. UNION ALL (اجتماع با تکرار)

همه سطرها را بدون حذف تکراری‌ها زیر هم می‌گذارد.

```sql
select s.[state], s.city, s.zip from dbo.stores as s
union all
select a.[state], a.city, a.zip from dbo.authors as a
```

در مثال مجموعه‌ها: `{10, 20, 30, 30, 50, 50, 90}`. چون نیازی به بررسی تکراری‌ها نیست، **سریع‌ترین** گزینه است. در کد شما ۳۷ سطر برمی‌گرداند.

> نکته: در کد شما اول `dunion all` نوشته شده که غلط تایپی است و باید `union all` باشد.

## ۲. UNION (اجتماع بدون تکرار)

مثل UNION ALL است، ولی سطرهای تکراری را حذف می‌کند (عملاً یک `DISTINCT` روی کل نتیجه).

```sql
select p.[state], p.city, NULL as zip from dbo.publishers as p
union
select s.[state], s.city, s.zip from dbo.stores as s
union
select a.[state], a.city, a.zip from dbo.authors as a
```

در مثال مجموعه‌ها: `{10, 20, 30, 50, 90}`. نتیجه در کد شما ۳۲ سطر است. چون SQL Server باید تکراری‌ها را پیدا کند (با sort یا hash)، **کندتر** از UNION ALL است. اگر مطمئنید تکراری وجود ندارد یا مهم نیست، از UNION ALL استفاده کنید.

## ساخت جدول جدید با SELECT INTO

```sql
select p.[state], p.city, NULL as zip into Locations
from dbo.publishers as p
union
select s.[state], s.city, s.zip from dbo.stores as s
union
select a.[state], a.city, a.zip from dbo.authors as a
```

عبارت `INTO` فقط در **Query اول** می‌آید و کل نتیجه ترکیب در جدول جدید `Locations` ذخیره می‌شود. ساختار جدول (نام و نوع ستون‌ها) از Query اول تعیین می‌شود. چون `NULL as zip` نوع مشخصی ندارد، بهتر است برای کنترل نوع از `CAST(NULL as char(5)) as zip` استفاده کنید.

## ۳. INTERSECT (اشتراک)

فقط سطرهایی را برمی‌گرداند که **در هر دو Query وجود دارند** (و تکراری‌ها را هم حذف می‌کند).

```sql
select p.[state], p.city from dbo.publishers as p
intersect
select a.[state], a.city from dbo.authors as a
```

یعنی شهر-ایالت‌هایی که هم ناشر دارند و هم نویسنده. در مثال مجموعه‌ها: A ∩ B = `{30, 50}`.

## ۴. EXCEPT (تفاضل)

سطرهای Query اول را برمی‌گرداند که **در Query دوم نیستند**. ترتیب Queryها مهم است.

```sql
select a.[state], a.city from dbo.authors as a   -- A
except
select p.[state], p.city from dbo.publishers as p -- B
```

یعنی شهرهایی که نویسنده دارند ولی ناشر ندارند. فرمول آن `A − (A ∩ B)` است. در مثال مجموعه‌ها: `{10}`.

نکته مهم: EXCEPT هم خروجی را **Distinct** می‌کند. برای همین در کامنت کد شما نوشته شده «۲۳ سطر ← ۲۱ سطر»: جدول authors ۲۳ سطر دارد، ولی بعد از حذف تکراری‌ها و کم کردن اشتراک، ۲۱ سطر می‌ماند. همچنین `A EXCEPT B` با `B EXCEPT A` فرق دارد.

## جمع‌بندی

| عملگر | معادل ریاضی | تکراری‌ها | سرعت |
|---|---|---|---|
| `UNION ALL` | A ∪ B (با تکرار) | حفظ می‌شوند | سریع‌ترین |
| `UNION` | A ∪ B | حذف می‌شوند | کندتر |
| `INTERSECT` | A ∩ B | حذف می‌شوند | کندتر |
| `EXCEPT` | A − B | حذف می‌شوند | کندتر |

## نکته درباره NULL

در این عملگرها (به جز عملکرد معمول `=` در WHERE) دو مقدار `NULL` **برابر در نظر گرفته می‌شوند**. یعنی اگر دو سطر در ستون‌های `NULL` یکسان باشند، UNION آن‌ها را یکی می‌کند و INTERSECT آن‌ها را مشترک می‌داند.

اگر بخواهید، می‌توانم چند تمرین روی دیتابیس pubs برایتان بنویسم، یا بخش بعدی یعنی `GUID` را که در انتهای کدتان نوشته‌اید توضیح بدهم.

**Example 6.1 — the instructor's sets, run for real.** `VALUES` builds a small in-memory table, so no database table is needed.

In [ ]:
SELECT x FROM (VALUES (10), (50), (30)) AS A(x)
INTERSECT
SELECT x FROM (VALUES (90), (20), (30), (50)) AS B(x)
ORDER BY x;                                   -- INTERSECT

SELECT x FROM (VALUES (10), (50), (30)) AS A(x)
EXCEPT
SELECT x FROM (VALUES (90), (20), (30), (50)) AS B(x)
ORDER BY x;                                   -- EXCEPT (A - B)

SELECT x FROM (VALUES (10), (50), (30)) AS A(x)
UNION ALL
SELECT x FROM (VALUES (90), (20), (30), (50)) AS B(x)
ORDER BY x;                                   -- UNION ALL

SELECT x FROM (VALUES (10), (50), (30)) AS A(x)
UNION
SELECT x FROM (VALUES (90), (20), (30), (50)) AS B(x)
ORDER BY x;                                   -- UNION

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>x</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>30</td>
    </tr>
    <tr>
      <td>50</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">2 row(s) — INTERSECT</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>x</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>10</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">1 row(s) — EXCEPT (A − B)</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>x</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>10</td>
    </tr>
    <tr>
      <td>20</td>
    </tr>
    <tr>
      <td>30</td>
    </tr>
    <tr>
      <td>30</td>
    </tr>
    <tr>
      <td>50</td>
    </tr>
    <tr>
      <td>50</td>
    </tr>
    <tr>
      <td>90</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">7 row(s) — UNION ALL</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>x</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>10</td>
    </tr>
    <tr>
      <td>20</td>
    </tr>
    <tr>
      <td>30</td>
    </tr>
    <tr>
      <td>50</td>
    </tr>
    <tr>
      <td>90</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">5 row(s) — UNION</p></div>

### The `pubs` example: states, cities and zip codes
Three tables hold a location: `publishers` (state, city), `stores` (state, city, zip) and `authors` (state, city, zip). Publishers have no zip code, so we supply a placeholder. First, how many rows does each table have?

In [ ]:
SELECT 'publishers' AS TableName, COUNT(*) AS NumRows FROM dbo.publishers
UNION ALL
SELECT 'stores',                  COUNT(*)           FROM dbo.stores
UNION ALL
SELECT 'authors',                 COUNT(*)           FROM dbo.authors;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>TableName</th>
      <th>NumRows</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>publishers</td>
      <td>8</td>
    </tr>
    <tr>
      <td>stores</td>
      <td>6</td>
    </tr>
    <tr>
      <td>authors</td>
      <td>23</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">3 row(s)</p></div>

**Example 6.2 — `UNION ALL`: everything, duplicates included.** The row count is simply the sum of the three tables: **37** in the instructor's notes. (The copy of the notes you shared has the first `SELECT` missing and the keyword mistyped as `dunion all`; see the fix list at the end.)

We write the missing zip as `CAST(NULL AS CHAR(5))` rather than a bare `NULL`. A bare `NULL` in the first query has type `INT`, and SQL Server would then try to convert the other queries' zip codes to integers, which strips leading zeros from zips such as `04101`.

In [ ]:
SELECT p.[state], p.city, CAST(NULL AS CHAR(5)) AS zip FROM dbo.publishers AS p   -- A
UNION ALL
SELECT s.[state], s.city, s.zip                        FROM dbo.stores     AS s   -- B
UNION ALL
SELECT a.[state], a.city, a.zip                        FROM dbo.authors    AS a;  -- C

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>state</th>
      <th>city</th>
      <th>zip</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>MA</td>
      <td>Boston</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>DC</td>
      <td>Washington</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Berkeley</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>IL</td>
      <td>Chicago</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>TX</td>
      <td>Dallas</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>NULL</td>
      <td>München</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>NY</td>
      <td>New York</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>NULL</td>
      <td>Paris</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>WA</td>
      <td>Seattle</td>
      <td>98056</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Tustin</td>
      <td>92789</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Los Gatos</td>
      <td>96745</td>
    </tr>
    <tr>
      <td>WA</td>
      <td>Remulade</td>
      <td>98014</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Fremont</td>
      <td>90019</td>
    </tr>
    <tr>
      <td>OR</td>
      <td>Portland</td>
      <td>89076</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Menlo Park</td>
      <td>94025</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Oakland</td>
      <td>94618</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Berkeley</td>
      <td>94705</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>San Jose</td>
      <td>95128</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Oakland</td>
      <td>94609</td>
    </tr>
    <tr>
      <td>KS</td>
      <td>Lawrence</td>
      <td>66044</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Berkeley</td>
      <td>94705</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Palo Alto</td>
      <td>94301</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Covelo</td>
      <td>95428</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>San Francisco</td>
      <td>94130</td>
    </tr>
    <tr>
      <td>TN</td>
      <td>Nashville</td>
      <td>37215</td>
    </tr>
    <tr>
      <td>OR</td>
      <td>Corvallis</td>
      <td>97330</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Walnut Creek</td>
      <td>94595</td>
    </tr>
    <tr>
      <td>MI</td>
      <td>Ann Arbor</td>
      <td>48105</td>
    </tr>
    <tr>
      <td>IN</td>
      <td>Gary</td>
      <td>46403</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Oakland</td>
      <td>94609</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Oakland</td>
      <td>94612</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Oakland</td>
      <td>94609</td>
    </tr>
    <tr>
      <td>MD</td>
      <td>Rockville</td>
      <td>20853</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Palo Alto</td>
      <td>94301</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Vacaville</td>
      <td>95688</td>
    </tr>
    <tr>
      <td>UT</td>
      <td>Salt Lake City</td>
      <td>84152</td>
    </tr>
    <tr>
      <td>UT</td>
      <td>Salt Lake City</td>
      <td>84152</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">37 row(s) — UNION ALL simply stacks the three result sets (publishers, then stores, then authors)</p></div>

**Example 6.3 — `UNION`: duplicates removed.** The same three queries with `UNION` return **32** rows in the instructor's notes: five fewer, because five (state, city, zip) combinations appeared more than once. Notice the alias `Shalgham` (Persian for *turnip*) on the second query's zip column. It is deliberately silly, to prove rule 3: the result's columns are named after the **first** query, so the output still says `zip`.

In [ ]:
SELECT p.[state], p.city, CAST(NULL AS CHAR(5)) AS zip      FROM dbo.publishers AS p   -- A
UNION
SELECT s.[state], s.city, s.zip AS Shalgham                 FROM dbo.stores     AS s   -- B
UNION
SELECT a.[state], a.city, a.zip                             FROM dbo.authors    AS a   -- C
ORDER BY [state], city;          -- ORDER BY only at the very end, using the FIRST query's names

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>state</th>
      <th>city</th>
      <th>zip</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>NULL</td>
      <td>München</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>NULL</td>
      <td>Paris</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Berkeley</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Berkeley</td>
      <td>94705</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Covelo</td>
      <td>95428</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Fremont</td>
      <td>90019</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Los Gatos</td>
      <td>96745</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Menlo Park</td>
      <td>94025</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Oakland</td>
      <td>94609</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Oakland</td>
      <td>94618</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Oakland</td>
      <td>94612</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Palo Alto</td>
      <td>94301</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>San Francisco</td>
      <td>94130</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>San Jose</td>
      <td>95128</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Tustin</td>
      <td>92789</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Vacaville</td>
      <td>95688</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Walnut Creek</td>
      <td>94595</td>
    </tr>
    <tr>
      <td>DC</td>
      <td>Washington</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>IL</td>
      <td>Chicago</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>IN</td>
      <td>Gary</td>
      <td>46403</td>
    </tr>
    <tr>
      <td>KS</td>
      <td>Lawrence</td>
      <td>66044</td>
    </tr>
    <tr>
      <td>MA</td>
      <td>Boston</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>MD</td>
      <td>Rockville</td>
      <td>20853</td>
    </tr>
    <tr>
      <td>MI</td>
      <td>Ann Arbor</td>
      <td>48105</td>
    </tr>
    <tr>
      <td>NY</td>
      <td>New York</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>OR</td>
      <td>Corvallis</td>
      <td>97330</td>
    </tr>
    <tr>
      <td>OR</td>
      <td>Portland</td>
      <td>89076</td>
    </tr>
    <tr>
      <td>TN</td>
      <td>Nashville</td>
      <td>37215</td>
    </tr>
    <tr>
      <td>TX</td>
      <td>Dallas</td>
      <td>NULL</td>
    </tr>
    <tr>
      <td>UT</td>
      <td>Salt Lake City</td>
      <td>84152</td>
    </tr>
    <tr>
      <td>WA</td>
      <td>Remulade</td>
      <td>98014</td>
    </tr>
    <tr>
      <td>WA</td>
      <td>Seattle</td>
      <td>98056</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">32 row(s) — the column is named zip, not Shalgham</p></div>

**Example 6.4 — combining set operators with `SELECT INTO`.** The instructor saves the combined list as a table called `Locations`. The rule: `INTO` goes in the **first** `SELECT` only, and the table receives the whole combined result.

In [ ]:
DROP TABLE IF EXISTS dbo.Locations_NB;

SELECT p.[state], p.city, CAST(NULL AS CHAR(5)) AS zip
INTO   dbo.Locations_NB                       -- INTO belongs to the first SELECT
FROM   dbo.publishers AS p
UNION
SELECT s.[state], s.city, s.zip FROM dbo.stores  AS s
UNION
SELECT a.[state], a.city, a.zip FROM dbo.authors AS a;

SELECT COUNT(*) AS LocationRows FROM dbo.Locations_NB;

(32 rows affected)

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>LocationRows</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>32</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">1 row(s)</p></div>

**Example 6.5 — `INTERSECT`: what appears in both?** Which (state, city) pairs exist both as a publisher location and as an author location? Only rows present in *both* result sets are kept.

In [ ]:
SELECT p.[state], p.city FROM dbo.publishers AS p    -- A
INTERSECT
SELECT a.[state], a.city FROM dbo.authors    AS a;   -- B

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>state</th>
      <th>city</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>CA</td>
      <td>Berkeley</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">1 row(s)</p></div>

**Example 6.6 — `EXCEPT`: in A but not in B, and why order matters.** `A EXCEPT B` equals `A − (A ∩ B)`: take everything from A and remove whatever also appears in B. Unlike the other operators, **swapping the two queries changes the result**.

`EXCEPT` also removes duplicates. That is why the second query returns **15 rows**, not 21: the 23 author rows contain only 16 distinct (state, city) pairs, and removing the one pair shared with a publisher (Berkeley, CA) leaves 15. The instructor's note, *23 Rows ⇒ 21 Rows*, matches a different calculation: 23 author rows minus the two Berkeley authors is 21. That would be the answer if duplicates were kept, but `EXCEPT` does not keep them. It is a good thing to verify live in class.

In [ ]:
-- Publisher locations that are NOT author locations (A - B)
SELECT p.[state], p.city FROM dbo.publishers AS p
EXCEPT
SELECT a.[state], a.city FROM dbo.authors    AS a;

-- Author locations that are NOT publisher locations (B - A)
SELECT a.[state], a.city FROM dbo.authors    AS a
EXCEPT
SELECT p.[state], p.city FROM dbo.publishers AS p;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>state</th>
      <th>city</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>NULL</td>
      <td>München</td>
    </tr>
    <tr>
      <td>NULL</td>
      <td>Paris</td>
    </tr>
    <tr>
      <td>DC</td>
      <td>Washington</td>
    </tr>
    <tr>
      <td>IL</td>
      <td>Chicago</td>
    </tr>
    <tr>
      <td>MA</td>
      <td>Boston</td>
    </tr>
    <tr>
      <td>NY</td>
      <td>New York</td>
    </tr>
    <tr>
      <td>TX</td>
      <td>Dallas</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">7 row(s) — A − B</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>state</th>
      <th>city</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>CA</td>
      <td>Covelo</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Menlo Park</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Oakland</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Palo Alto</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>San Francisco</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>San Jose</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Vacaville</td>
    </tr>
    <tr>
      <td>CA</td>
      <td>Walnut Creek</td>
    </tr>
    <tr>
      <td>IN</td>
      <td>Gary</td>
    </tr>
    <tr>
      <td>KS</td>
      <td>Lawrence</td>
    </tr>
    <tr>
      <td>MD</td>
      <td>Rockville</td>
    </tr>
    <tr>
      <td>MI</td>
      <td>Ann Arbor</td>
    </tr>
    <tr>
      <td>OR</td>
      <td>Corvallis</td>
    </tr>
    <tr>
      <td>TN</td>
      <td>Nashville</td>
    </tr>
    <tr>
      <td>UT</td>
      <td>Salt Lake City</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">15 row(s) — B − A: 23 author rows shrink to 16 distinct pairs, minus Berkeley = 15</p></div>

**Example 6.7 — precedence, and the errors you will meet.** With `UNION` followed by `INTERSECT`, SQL Server evaluates `INTERSECT` first: A ∪ (B ∩ C). Parentheses change the meaning to (A ∪ B) ∩ C. The classic error (queries with a different number of columns) follows in its own cell, because SQL Server rejects the whole batch at compile time.

In [ ]:
-- A UNION (B INTERSECT C)  ->  INTERSECT is evaluated first
SELECT x FROM (VALUES (1), (2), (3)) AS A(x)
UNION
SELECT x FROM (VALUES (2), (3), (4)) AS B(x)
INTERSECT
SELECT x FROM (VALUES (3), (4), (5)) AS C(x)
ORDER BY x;

-- (A UNION B) INTERSECT C  ->  parentheses force UNION first
(SELECT x FROM (VALUES (1), (2), (3)) AS A(x)
 UNION
 SELECT x FROM (VALUES (2), (3), (4)) AS B(x))
INTERSECT
SELECT x FROM (VALUES (3), (4), (5)) AS C(x)
ORDER BY x;

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>x</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>1</td>
    </tr>
    <tr>
      <td>2</td>
    </tr>
    <tr>
      <td>3</td>
    </tr>
    <tr>
      <td>4</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">4 row(s) — A ∪ (B ∩ C) = {1,2,3} ∪ {3,4}</p></div>

<div><style scoped>.dataframe tbody tr th:only-of-type{vertical-align:middle}.dataframe tbody tr th{vertical-align:top}.dataframe thead th{text-align:right}</style><table border="1" class="dataframe dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>x</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>1</td>
    </tr>
    <tr>
      <td>2</td>
    </tr>
    <tr>
      <td>3</td>
    </tr>
    <tr>
      <td>4</td>
    </tr>
  </tbody>
</table><p style="color:gray;font-size:90%;margin:4px 0 0 0">4 row(s) — A ∪ (B ∩ C) = {1,2,3} ∪ {3,4}</p></div>

In [ ]:
-- Different number of columns
SELECT p.[state], p.city FROM dbo.publishers AS p
UNION
SELECT s.[state]         FROM dbo.stores     AS s;

Msg 205, Level 16, State 1, Line 1
All queries combined using a UNION, INTERSECT or EXCEPT operator must have an equal number of expressions in their target lists.

### Pitfalls — set operators
- Different column counts or incompatible types raise an error; matching is by *position*, so a swapped column order gives wrong results without any error.
- `UNION` quietly removes duplicates, which also costs a sort or hash. If you expected 37 rows and got 32, that is why.
- A bare `NULL` as the first query's column fixes the type to `INT`. Use `CAST(NULL AS ...)`.
- `EXCEPT` is directional. `INTERSECT` and `EXCEPT` can also be written with `EXISTS` or joins, but the set operators read more clearly and treat `NULL`s as equal.

---
## 7. Wrap-up

### 7.1 Clean-up
Removes the tables this notebook created (all end in `_NB`, so your classroom tables are untouched).

In [ ]:
DROP TABLE IF EXISTS dbo.Sematec_Books_NB;
DROP TABLE IF EXISTS dbo.Sematec_Books_Empty_NB;
DROP TABLE IF EXISTS dbo.MyCustomerCopy_NB;
DROP TABLE IF EXISTS dbo.CustomerSample_NB;
DROP TABLE IF EXISTS dbo.Locations_NB;

Commands completed successfully.

### 7.2 Cheat sheet

| Goal | Pattern |
|---|---|
| Sort ascending / descending | `ORDER BY col [ASC]` / `ORDER BY col DESC` |
| Sort with a tie-breaker | `ORDER BY price DESC, title ASC` |
| Show missing values as zero | `ISNULL(price, 0) AS BookPrice` |
| Highest-priced row | `SELECT TOP 1 ... ORDER BY price DESC` |
| Top *n* including ties | `SELECT TOP (n) WITH TIES ... ORDER BY ...` |
| First *p* percent | `SELECT TOP (p) PERCENT ...` (rounds up) |
| Paging | `ORDER BY ... OFFSET k ROWS FETCH NEXT n ROWS ONLY` |
| New GUID | `NEWID()` |
| Random *n* rows | `SELECT TOP (n) ... ORDER BY NEWID()` |
| Auto-increment key | `col INT IDENTITY(1,1)` |
| Fully qualified name | `database.schema.table` (`pubs.dbo.titles`) |
| Save a result as a table | `SELECT ... INTO new_table FROM ...` |
| Copy only the structure | `SELECT ... INTO new_table FROM ... WHERE 1 = 0` |
| Stack results, keep duplicates | `query1 UNION ALL query2` |
| Stack results, remove duplicates | `query1 UNION query2` |
| Rows in both | `query1 INTERSECT query2` |
| Rows in the first only | `query1 EXCEPT query2` |

### 7.3 Small fixes for your copy of the instructor's notes
1. **`dunion all`** (in the first set-operator example) is a typo for `union all`, and the `publishers` query that should precede it is missing. Without it, you would get 29 rows (6 stores + 23 authors), not the 37 recorded in the comment.
2. **`select NEWSEQUENTIALID()`** raises an error when run on its own. The function is valid only inside a `DEFAULT` constraint (Example 3.5).
3. **`DISTINT`** in the comment skeleton should be `DISTINCT`.
4. **`select top 5 ...` without `ORDER BY`** returns an arbitrary five rows. This is fine as a demonstration, but never rely on it.
5. **`NULL AS zip`** as the first query's third column makes the combined column an integer type. `CAST(NULL AS CHAR(5))` is safer, especially before `SELECT ... INTO Locations`.
6. **`order by t.price desc, title asc`**: qualify it as `t.title` to stay consistent.
7. **`DATEDIFF(year, ...)`** gives an approximate age (see Example 3.2).
8. **`use DB1 ... select * into pubs..MyTable from SampleTable`** only runs if the database `DB1` and the table `SampleTable` exist on your server.
9. **`ORDER BY` inside `SELECT INTO`** does not guarantee the order in which the table will later return its rows.
10. **`23 Rows => 21 Rows`** (the `EXCEPT` example) does not match what `EXCEPT` returns on Microsoft's `pubs` data: 15 rows, because `EXCEPT` removes duplicates (Example 6.6).

### 7.4 Practice exercises
Try these on your own before looking at anything else; each has a one-line hint.

1. List the five cheapest books that actually have a price, ties included. *(`WHERE price IS NOT NULL`, `TOP ... WITH TIES`, `ORDER BY price`.)*
2. Show every book type once, sorted alphabetically, and then only the first two. *(`DISTINCT` + `TOP` + `ORDER BY`.)*
3. From `DimCustomer`, return a random 5 percent sample, with the full name and the exact age. *(`TOP 5 PERCENT`, `ORDER BY NEWID()`, `CONCAT_WS`.)*
4. Save that sample as a table in `pubs`, add a primary key on `CustomerKey`, and check the key exists. *(`SELECT INTO`, `ALTER TABLE`, `sys.key_constraints`.)*
5. Which states have an author but neither a store nor a publisher? *(`EXCEPT`, applied twice or with `UNION` in parentheses.)*
6. Which (state, city) pairs appear in all three of `publishers`, `stores` and `authors`? *(`INTERSECT` twice.)*
7. Predict, then check: how many rows does `TOP 4 WITH TIES` return on the price ranking, and why? *(Look at Example 2.8.)*
8. Rewrite Example 6.3 so the ordering is by `city` descending, and explain why the `ORDER BY` can use the name `city` but not the alias `Shalgham`.